# Weekly Analytics Brief — OLJ + OT

One notebook builds **both** weekly briefs: a separate one-page Word (+ PDF) and a separate email for each
brand. Everything brand-specific lives in one place (the **Brands** section). Every other step is a shared
function that takes a brand, so there is no duplicated code.

| Step | What it does | Brands |
|---|---|---|
| ⚙️ Settings | CMS key, the acquisition rule, which brands to build, recipients per brand | — |
| 1. New accounts | `GET /cms/customer` | both at once |
| 2. New subscriptions | `GET /cms/payment` + each buyer's history | both at once |
| 3. GA4 | shared query functions (users, sessions, page views, articles, countries, sources, downloads) | per brand |
| 4. Brands | the `Brand` definitions, then one GA4 pull per brand | per brand |
| 5. Report | builds and previews each brief | per brand |
| 6. Export | one Word + PDF per brand | per brand |
| 7. Email | one email per brand, to that brand's recipients | per brand |

No spreadsheet is read. Churn is not reported for now. All rules and edge cases are in **Algo Assumptions**.
Run top to bottom.

## Dependencies

Run this once per Colab session (installs are wiped when the runtime resets).

In [ ]:
!pip install -q google-api-python-client google-auth google-auth-oauthlib google-analytics-data python-docx requests

## ⚙️ Settings — paste your CMS API key here

This is the only cell you should need to edit. **On GitHub** the key is read from the repo secret
`CMS_API_KEY` automatically, so nothing needs pasting there. **Easiest in Colab:** click the 🔑 **Secrets** icon in the
left sidebar, add `CMS_API_KEY`, and switch on notebook access. Then the key survives every new copy
of this notebook and you never paste it again. ⚠️ If this notebook ever goes to GitHub, leave the key
**empty** there and add a repo secret called `CMS_API_KEY` instead. The notebook picks it up automatically,
and a key committed to a repo is a leaked key.

In [ ]:
import os

# ---- CMS (new accounts + new subscriptions, both brands) --------------------------------------------------------------------
CMS_API_KEY = ""   # <-- paste your CMS API key between the quotes

def _colab_secret(name):
    """Colab's 🔑 Secrets panel (left sidebar) -- survives new copies of the notebook, unlike a pasted key."""
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

# Where the key comes from, first match wins:
#   1. GitHub secret CMS_API_KEY  (the workflow passes it in as an env var; nothing to do in the notebook)
#   2. Colab 🔑 Secret CMS_API_KEY
#   3. the line pasted above
_pasted_key = CMS_API_KEY.strip()
for _src, _val in (("GitHub secret / env var", os.environ.get("CMS_API_KEY", "")),
                   ("Colab secret", _colab_secret("CMS_API_KEY")),
                   ("pasted in this cell", _pasted_key)):
    if _val.strip():
        CMS_API_KEY, CMS_KEY_SOURCE = _val.strip(), _src
        break
else:
    CMS_API_KEY, CMS_KEY_SOURCE = "", "none"

if os.environ.get("GITHUB_ACTIONS") == "true" and _pasted_key:
    print("⚠️  A CMS key is pasted in this notebook and it's in the GitHub repo -- delete it from the cell and "
          "rotate the key in the CMS backend; the GitHub secret is all the workflow needs.")

# WhiteBeard's URL format is https://api.{your_domain}/cms/{path} -> for OLJ: https://api.lorientlejour.com/cms/customer
CMS_API_HOST = "api.lorientlejour.com"
CMS_API_HOST = (CMS_API_HOST or os.environ.get("CMS_API_HOST", "")).strip().removeprefix("https://").removeprefix("http://").strip("/")
CMS_BASE_URL = f"https://{CMS_API_HOST}" if CMS_API_HOST else ""

# The backend's /revenue/customer page URL. These columns are all the new-accounts count needs (keeps pages small).
# To make the CMS filter by creation date (much faster): apply the creation-date filter on that page,
# copy the URL, paste it here and replace the two dates with {start} and {end}. The notebook fills them
# in with last Monday -> this Sunday.
CMS_BACKEND_URL = (
    "https://managecmsnew.lorientlejour.com/revenue/customer?"
    "columns%5B%5D=id&columns%5B%5D=creationDate&columns%5B%5D=preferredLanguage&columns%5B%5D=source"
    "&creationDate_operator=between&creationDate%5Bstart%5D={start}&creationDate%5Bend%5D={end}"
)
CMS_DATE_FMT = "%Y-%m-%d"   # how {start}/{end} are written -- match whatever the backend URL uses

# ╔══════════════════════════════════════════════════════════════════════════════════════╗
# ║  NEW SUBSCRIPTIONS -- someone whose previous subscription ended at least LAPSE_DAYS       ║
# ║  days before their new order counts as a NEW acquisition. Change the number here.        ║
LAPSE_DAYS = 30
# ║  True = a payment that extends a subscription started long ago (e.g. Cash renewals       ║
# ║  entered by staff) is a renewal, not a new subscriber.                                   ║
OLD_SUB_EXTENSION_IS_RENEWAL = True
# ║  False (official rule) = a plan change never counts, even if the old subscription was    ║
# ║  cancelled and just running out; only a gap of LAPSE_DAYS+ makes someone new again.       ║
# ║  True = count those as NEW (the sheet does this in practice for a few orders a week).    ║
CANCELLED_IS_CHURNED = False
# ║  True = OLJ and OT are judged separately: someone with a running OLJ subscription who    ║
# ║  buys OT is a new OT subscriber (and vice versa). Also what the sheet does.              ║
BRANDS_SEPARATE = True
# ║  SIMPLE_RULE = True: count EVERY order from the CMS "New subscriptions" filter, except     ║
# ║  status Fail, amount 0, or donation / intégrale in the product. No buyer-history checks,   ║
# ║  every order counts (no one-per-person). False = the full rules above.                     ║
SIMPLE_RULE = True
# ║  With SIMPLE_RULE: orders whose "acquisition source" is Renewal are not counted (only New). ║
USE_ACQUISITION_SOURCE = True
# ╚══════════════════════════════════════════════════════════════════════════════════════╝

# ---- Acquisitions sheet (Google Sheet) -- its totals are shown in ( ) next to the new subscriptions ----
# Read with the same service account as GA4 (service_account.json; on GitHub written from the GSHEET_SA_KEY secret).
# The service account's e-mail must have Viewer access to the sheet (Share button in the Google Sheet).
ACQ_SHEET_ID = "11WU-b3nmvyPlO0fX9VycKgObr-v5-hXTN6ieCv2TOoA"
ACQ_SHEET_GID = 2059832827          # the tab from the link (#gid=...); falls back to a tab named "Acquisitions"
ACQ_SHEET_XLSX = "Daily_sheet_Acquisitions_Churns.xlsx"   # optional fallback: the sheet downloaded and uploaded in Colab

# ---- Brands to build ------------------------------------------------------------------------
BUILD_BRANDS = ["OLJ", "OT"]      # remove one to build a single brief

# ---- Email recipients ---------------------------------------------------------------------
# Test runs (you in Colab, or a manual "Run workflow" on GitHub) only go to TEST_RECIPIENTS -- both briefs.
TEST_RECIPIENTS = ["dianafarhat@lorientlejour.com"]
# The scheduled Monday 12:00 send goes to each brand's list. You can also manage them without touching the
# notebook: GitHub repo -> Settings -> Secrets and variables -> Actions -> Variables ->
# BRIEF_RECIPIENTS_OLJ / BRIEF_RECIPIENTS_OT (comma-separated); when set, they replace the lists below.
SCHEDULED_RECIPIENTS = {
    "OLJ": ["dianafarhat@lorientlejour.com"],
    "OT":  ["dianafarhat@lorientlejour.com"],
}
EMAIL_AS_BCC = False   # True = recipients don't see each other (and can't reply-all)

print("CMS key:", f"set (from {CMS_KEY_SOURCE})" if CMS_API_KEY else "MISSING", "| API:", (CMS_BASE_URL + "/cms/customer") if CMS_BASE_URL else "host MISSING")

In [2]:
import pandas as pd
import datetime as dt

# --- Week window ---
# Auto-detects the most recently COMPLETED Monday-Sunday week based on today's date.
# To check a specific past week instead, uncomment the override line below and edit it.
today = dt.datetime.combine(dt.date.today(), dt.time.min)
WEEK_END = today - dt.timedelta(days=today.weekday() + 1)  # most recent Sunday before today
# WEEK_END = dt.datetime(2026, 9, 20)  # <- uncomment + edit to override with a specific week

WEEK_START = WEEK_END - dt.timedelta(days=6)  # the Monday of that week
PREV_WEEK_END = WEEK_START - dt.timedelta(days=1)
PREV_WEEK_START = PREV_WEEK_END - dt.timedelta(days=6)

assert WEEK_END.weekday() == 6, "WEEK_END must be a Sunday"

print(f"This week:  {WEEK_START:%Y-%m-%d} (Mon) → {WEEK_END:%Y-%m-%d} (Sun)")
print(f"Last week:  {PREV_WEEK_START:%Y-%m-%d} (Mon) → {PREV_WEEK_END:%Y-%m-%d} (Sun)")


This week:  2026-09-14 (Mon) → 2026-09-20 (Sun)
Last week:  2026-09-07 (Mon) → 2026-09-13 (Sun)


## 1. New accounts — CMS (OLJ and OT, Web / App / Other)

Pulls every account created **last Monday → this Sunday** from `GET /cms/customer` (key and URLs come
from the ⚙️ Settings cell), then classifies each one:

| Field | Rule |
|---|---|
| **Brand** (`preferredLanguage`) | `en` / `english` / `anglais` → **OT** · anything else (blank, `fr`, `french`…) → **OLJ** |
| **Platform** (`source`) | `web_olj`, `web_ot` → **Web** · `app`, `todayapp` → **App** · anything else → **Other** (newsletters etc.) |

- In the scorecard, *New accounts* gets Web and App columns. "Other" is counted in the Total and noted
  under the row label, so Web + App + Other = Total.
- `creationDate` is UTC, so it's converted to **Beirut time** before bucketing into Mon–Sun weeks.
- Counting is always re-checked locally by date, so a missing or wrong server-side filter can't skew it.
  Without a date filter in `CMS_BACKEND_URL` it just pages through more customers (and stops early if
  results come newest-first).
- Each run prints which `source` and `preferredLanguage` values it saw. That makes any value landing in
  "Other" obvious, and you can add it to `SOURCE_PLATFORM` / `OT_LANGUAGES` below.
- If the CMS isn't configured or the call fails, the cell **stops with an error** (there's no sheet
  fallback any more), so a wrong number is never sent silently.

In [ ]:
import requests
from collections import Counter
from urllib.parse import urlsplit, parse_qsl
from zoneinfo import ZoneInfo

# ---------- Classification rules (edit here if new values show up) ----------
OT_LANGUAGES = {"en", "eng", "english", "anglais"}          # -> OT; anything else (blank, fr, french...) -> OLJ
SOURCE_PLATFORM = {"web_olj": "Web", "web_ot": "Web",       # -> Web
                   "app": "App", "todayapp": "App"}         # -> App; anything else -> Other
OTHER_BUCKET = "Other"
ACCOUNT_BUCKETS = ["Web", "App", OTHER_BUCKET]

CMS_TZ = ZoneInfo("Asia/Beirut")
CMS_MAX_PAGES = 2000        # safety stop
CMS_READY = bool(CMS_API_KEY and CMS_BASE_URL)

def _cms_check(resp):
    """Raise a clear error when Cloudflare (not the CMS) blocks the request, otherwise the usual HTTP error."""
    if resp.status_code in (403, 503) and "just a moment" in resp.text[:2000].lower():
        raise RuntimeError(
            "[CMS] Blocked by Cloudflare's bot check, not by the CMS -- the API key is fine.\n"
            "  Colab/GitHub machine was flagged. Try Runtime -> Disconnect and delete runtime, then rerun.\n"
            "  Permanent fix: ask IT to let /cms/* requests with an API-Key header skip the Cloudflare challenge.")
    resp.raise_for_status()

def _pick(c, *names):
    """First non-empty value for any of `names` (case-insensitive), looking at the top level and then
    inside `fields` / `preferences` -- the API and the backend page don't always name/nest columns alike."""
    wanted = {n.lower() for n in names}
    for scope in (c, c.get("fields"), c.get("preferences")):
        if isinstance(scope, dict):
            for k, v in scope.items():
                if k.lower() in wanted and v not in (None, ""):
                    return v
    return None

def _lang(c):
    return str(_pick(c, "preferredLanguage", "preferred_language", "language") or "").strip().lower()

def _source(c):
    return str(_pick(c, "source", "acquisitionSource", "acquisition_source") or "").strip().lower()

def brand_of(c):
    lang = _lang(c)
    return "ot" if lang in OT_LANGUAGES or lang.startswith(("en-", "en_")) else "olj"

def platform_of(c):
    return SOURCE_PLATFORM.get(_source(c), OTHER_BUCKET)

def _created_local(c):
    """creationDate (UTC ISO string) -> naive Beirut-time datetime, or None."""
    raw = _pick(c, "creationDate", "creation_date", "created")
    if not raw:
        return None
    try:
        d = dt.datetime.fromisoformat(str(raw).replace("Z", "+00:00"))
    except ValueError:
        return None
    if d.tzinfo is None:
        d = d.replace(tzinfo=dt.timezone.utc)
    return d.astimezone(CMS_TZ).replace(tzinfo=None)

def _backend_params(start, end):
    query = urlsplit(CMS_BACKEND_URL).query if "?" in CMS_BACKEND_URL else CMS_BACKEND_URL
    fill = lambda v: v.replace("{start}", start.strftime(CMS_DATE_FMT)).replace("{end}", end.strftime(CMS_DATE_FMT))
    params = [(k, fill(v)) for k, v in parse_qsl(query, keep_blank_values=True) if k != "page" and v != ""]
    has_date_filter = any("{start}" in v for _, v in parse_qsl(query))   # parse_qsl decodes %7Bstart%7D too
    return params, has_date_filter

def cms_fetch_customers(start, end):
    """Every customer the query returns, across all pages (deduped by id). Stops early when there's no
    server-side date filter but results come newest-first and have gone past `start`."""
    base_params, has_date_filter = _backend_params(start, end)
    if not has_date_filter:
        print("[CMS] no creation-date filter in CMS_BACKEND_URL -- fetching and filtering here (slower)")

    session = requests.Session()
    session.headers.update({"API-Key": CMS_API_KEY, "Accept": "application/json"})
    url = CMS_BASE_URL.rstrip("/") + "/cms/customer"

    customers, seen, page = [], set(), None
    newest_first, prev_last = True, None
    for _ in range(CMS_MAX_PAGES):
        params = base_params + ([("page", page)] if page is not None else [])
        resp = session.get(url, params=params, timeout=60)
        _cms_check(resp)
        payload = resp.json()
        batch = []
        for c in payload.get("data") or []:
            key = str(_pick(c, "userId", "id") or id(c))
            if key not in seen:
                seen.add(key); batch.append(c)
        if not batch:                                   # empty or repeated page -> done
            break
        customers += batch
        total = payload.get("total") or 0
        if total and len(seen) >= total:
            break

        dates = [d for d in map(_created_local, batch) if d]
        if dates:
            chain = ([prev_last] if prev_last else []) + dates
            newest_first = newest_first and all(a >= b for a, b in zip(chain, chain[1:]))
            prev_last = dates[-1]
            if not has_date_filter and newest_first and dates[-1] < start:
                break                                   # sorted newest-first and already past the window

        page = (payload["page"] if payload.get("page") is not None else (page or 0)) + 1
    else:
        print(f"[CMS] stopped after {CMS_MAX_PAGES} pages -- add the creation-date filter to CMS_BACKEND_URL")
    return customers

def cms_accounts_week(customers, start, end):
    """{'olj_new_accounts', 'ot_new_accounts', 'olj_split': {'Web','App','Other'}, 'ot_split': {...}}"""
    counts = {"olj": Counter(), "ot": Counter()}
    for c in customers:
        d = _created_local(c)
        if d is not None and start.date() <= d.date() <= end.date():
            counts[brand_of(c)][platform_of(c)] += 1
    out = {}
    for b in ("olj", "ot"):
        out[f"{b}_split"] = {k: counts[b].get(k, 0) for k in ACCOUNT_BUCKETS}
        out[f"{b}_new_accounts"] = sum(out[f"{b}_split"].values())
    return out

# ---------- Build this week / last week ----------
if not CMS_READY:
    raise RuntimeError("CMS not configured -- fill in CMS_API_KEY in ⚙️ Settings (or the Colab / GitHub secret).")
try:
    # one pull covers both weeks; starts a day early because the CMS filters by UTC date while we bucket in
    # Beirut time (a Sunday-night-UTC account is Monday in Beirut). The local date check below drops the extra day.
    _customers = cms_fetch_customers(PREV_WEEK_START - dt.timedelta(days=1), WEEK_END)
except requests.HTTPError as e:
    raise RuntimeError(f"[CMS] new accounts failed (401/403 -> check CMS_API_KEY): {e}") from e
this_week_accounts = cms_accounts_week(_customers, WEEK_START, WEEK_END)
last_week_accounts = cms_accounts_week(_customers, PREV_WEEK_START, PREV_WEEK_END)
ACCOUNTS_SOURCE = "CMS"

_in_window = [c for c in _customers if (d := _created_local(c)) and PREV_WEEK_START.date() <= d.date() <= WEEK_END.date()]
NEW_ACCOUNT_IDS = {str(i) for c in _in_window if (i := _pick(c, "userId", "id")) is not None}
print(f"[CMS] {len(_customers)} fetched, {len(_in_window)} created in the two weeks")
print("  source values:  ", dict(Counter(_source(c) or "(blank)" for c in _in_window).most_common()))
print("  language values:", dict(Counter(_lang(c) or "(blank)" for c in _in_window).most_common()))
if _customers and not _in_window:
    print("  first record's fields (check the names match):", sorted(_customers[0].keys()))
this_week_accounts, last_week_accounts


## 2. New subscriptions (acquisitions) — OLJ and OT, orders + buyer history

1. **Orders:** `GET /cms/payment`, the API behind the backend's `/revenue/order` page, with its "new
   subscriptions" filters: paid, dated last Monday → this Sunday, subscription orders only
   (`hasRecurrentItems=1`), not automatic renewals (`referenceOrderId` empty).
2. **History:** `GET /cms/customer/{id}/subscription` for each buyer. Each order gets one label, checked in
   this order:

| Label | Rule | Counted? |
|---|---|---|
| excluded: donation / integrale | group contains donation or intégrale | no |
| excluded: free (staff) / test | payment method Free or a test gateway | no |
| excluded: paid 0 (free / staff) | the order's amount is 0 (e.g. a staff account entered as Cash 0$) | no |
| not new: renewal of an existing subscription | the order extends a subscription that started 7+ days earlier (`OLD_SUB_EXTENSION_IS_RENEWAL`) | no |
| not new: still has a valid free account | a free/staff subscription is still valid on the order date | no |
| NEW: brand new | no earlier subscription at all | **yes** |
| NEW: had only an expired free account | only earlier free accounts, all expired (e.g. ex-employee now paying) | **yes** |
| NEW: had cancelled, came back before it ran out | the earlier subscription was already cancelled / expiring (not closed by this order), `CANCELLED_IS_CHURNED` | **yes** |
| NEW: new to this brand | the only running subscription is the other brand (`BRANDS_SEPARATE`) | **yes** |
| not new: still subscribed (or changed plan) | an earlier paid subscription (same brand when `BRANDS_SEPARATE`) hadn't ended at the order date, or was closed by this very order | no |
| NEW: came back after N+ days | every earlier paid subscription ended `LAPSE_DAYS`+ days before (30 by default) | **yes** |
| not new: came back < N days | the last one ended less than `LAPSE_DAYS` days before | no |

3. **Count:** brand from the group (English / OT → **OT**, else **OLJ**), platform from the payment method
   (Apple / iTunes / Google → **App**, else **Web**); one per person per brand per week.

It prints the label counts, the totals, and a cross-check against the CMS's own new-orders count
(`/cms/analytics/orders/count-by-payment-method`, `order_type=2`) — the two order counts should be equal.
One line per order is saved to `acquisitions_orders.csv` for spot checks.

In [ ]:
import calendar, csv, re, unicodedata
from pathlib import Path

# ---------- Rules (the two main settings are in ⚙️ Settings: LAPSE_DAYS, OLD_SUB_EXTENSION_IS_RENEWAL) ----------
EXCLUDED_GROUP_WORDS = ("donation", "integrale")          # accents ignored -> "intégrale" matches too
OT_GROUP_WORDS = ("english",)                             # + the word "OT" -> OT; any other group -> OLJ
APP_METHOD_WORDS = ("apple", "itunes", "app store", "appstore", "ios", "google", "play store", "playstore", "android")
EXCLUDED_METHOD_WORDS = ("free", "test")                  # staff accounts / test payments
RENEWAL_GAP_DAYS = 7              # order's subscription started more than this long before the order -> an extension
ACTIVE_CODES = {1, 2, 4, 7}       # CMS status_code: active, grace period, pending user cancellation, renewal disabled
ENDED_CODES = {3, 5, 6, 9}        # expired, user cancelled, on hold, upgraded -> ended even without a deactivation date
NOT_STARTED = 8
ACQ_CSV = Path("acquisitions_orders.csv")   # one line per order with its decision (ids and dates only)

_acq = requests.Session()
_acq.headers.update({"API-Key": CMS_API_KEY, "Accept": "application/json"})

def _acq_get(path, params=None):
    r = _acq.get(CMS_BASE_URL.rstrip("/") + path, params=params, timeout=60)
    _cms_check(r)
    return r.json()

def _anorm(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().strip().lower()

def _aval(o, *keys, inner=("name", "title", "label", "description")):
    """First non-empty value for any key (case-insensitive); objects -> their name."""
    if not isinstance(o, dict):
        return None
    for k in keys:
        for kk, v in o.items():
            if kk.lower() == k.lower() and v not in (None, ""):
                if isinstance(v, dict):
                    v = next((v[n] for n in inner if v.get(n)), None)
                if v is not None and not isinstance(v, list):
                    return v
    return None

def _adt(raw):
    """CMS date string -> naive datetime (as shown in the backend); '0000-00-00' / blank -> None."""
    if raw in (None, "") or str(raw).startswith("0000"):
        return None
    try:
        d = dt.datetime.fromisoformat(str(raw).strip().replace("Z", "+00:00"))
    except ValueError:
        return None
    return d.astimezone(CMS_TZ).replace(tzinfo=None) if d.tzinfo else d

def _add_months(d, m):
    y, mo = divmod(d.month - 1 + m, 12); y, mo = d.year + y, mo + 1
    return d.replace(year=y, month=mo, day=min(d.day, calendar.monthrange(y, mo)[1]))

def acq_brand(group):
    g = _anorm(group or "")
    if any(w in g for w in EXCLUDED_GROUP_WORDS):
        return None
    return "ot" if re.search(r"\bot\b", g) or any(w in g for w in OT_GROUP_WORDS) else "olj"

def acq_platform(method):
    m = _anorm(method or "")
    if any(w in m for w in EXCLUDED_METHOD_WORDS):
        return None
    return "App" if any(w in m for w in APP_METHOD_WORDS) else "Web"

def _week_name(d):
    if d is None:
        return None
    if WEEK_START.date() <= d.date() <= WEEK_END.date():
        return "this"
    if PREV_WEEK_START.date() <= d.date() <= PREV_WEEK_END.date():
        return "last"
    return None

# ---------- Step 1: new subscription orders in the two weeks ----------
def fetch_new_orders():
    """GET /cms/payment with the backend /revenue/order filters: paid, dated in the two weeks, subscription
    orders only, not renewals (referenceOrderId empty)."""
    base = ([] if SIMPLE_RULE else [("status", "paid")]) + [("date_operator", "between"),
            ("date[start]", f"{PREV_WEEK_START:%Y-%m-%d}"), ("date[end]", f"{WEEK_END:%Y-%m-%d}"),
            ("hasRecurrentItems", "1"), ("referenceOrderId_operator", "eq"), ("referenceOrderId", "")] + \
           [("columns[]", c) for c in ("id", "user_id", "group", "paymentMethod", "date", "status", "referenceOrderId", "amount", "products", "acquisitionSource")]
    rows, seen, page = [], set(), None
    for _ in range(CMS_MAX_PAGES):
        body = _acq_get("/cms/payment", base + ([("page", page)] if page else []))
        fresh = [o for o in (body.get("data") or []) if str(o.get("id")) not in seen]
        if not fresh:
            break
        seen.update(str(o.get("id")) for o in fresh); rows += fresh
        page = (body["page"] if body.get("page") is not None else (page or 0)) + 1
    return rows

# ---------- Step 2: each buyer's history -> new or not ----------
def _sub(s):
    return {"id": str(s.get("id")), "start": _adt(_aval(s, "activationDate")), "end": _adt(_aval(s, "deactivationDate")),
            "renew": _adt(_aval(s, "nextRenewal")), "code": _aval(s, "status_code"),
            "ref": str(_aval(s, "referenceOrderId") or ""), "group": _aval(s, "group_name", "groupName", "group"),
            "method": _aval(s, "paymentMethod")}

def _code(p):
    try:
        return int(p["code"])
    except (TypeError, ValueError):
        return None

def _ended_at(p):
    if p["end"]:
        return p["end"]
    return p["renew"] if _code(p) in ENDED_CODES else None          # None = still running

def _is_free(p):
    return "free" in _anorm(p["group"] or "") or acq_platform(p["method"]) is None

def _valid_at(p, when):
    if p["end"]:
        return p["end"] > when and p["end"] > (p["start"] or when)
    if _code(p) in ACTIVE_CODES:
        return not (p["renew"] and p["renew"] < when)
    return bool(p["renew"] and p["renew"] > when)

CANCEL_CODES = {3, 4, 5}          # expired, pending user cancellation, user cancelled
CLOSED_BY_ORDER_MINUTES = 10      # old subscription closed within this of the new order = plan change / renewal

def _cancelled_before(p, when):
    """Old subscription was already on its way out (cancelled / expiring) and NOT closed by this new order."""
    stop = p["end"] or p["renew"]
    return (CANCELLED_IS_CHURNED and _code(p) in CANCEL_CODES and stop is not None
            and abs((stop - when).total_seconds()) > CLOSED_BY_ORDER_MINUTES * 60)

def _amount(order):
    """Amount paid as a number, or None if the API didn't send it (then the order is NOT excluded)."""
    raw = _aval(order, "amount", "total", "totalAmount", "price")
    try:
        return float(str(raw).replace(",", "").strip())
    except (TypeError, ValueError):
        return None

OT_PRODUCT_PATTERN = re.compile(r"today|\bot\b")   # "L'Orient Today (...)", "OT ..." -> OT, whatever the group says

def _product_text(order):
    """The order's product name(s) as one string -- the API may send text, an object or a list of objects."""
    for k, v in order.items():
        if k.lower() in ("products", "product", "productname", "product_name", "items"):
            items = v if isinstance(v, list) else [v]
            names = [(i.get("name") or i.get("title") or i.get("label") or "") if isinstance(i, dict) else str(i or "")
                     for i in items]
            return " | ".join(n for n in names if n)
    return ""

def order_brand(group, product):
    """None = excluded (donation / integrale in the product or the group). Product decides OT first, then the group."""
    p = _anorm(product or "")
    if any(w in p for w in EXCLUDED_GROUP_WORDS):
        return None
    if OT_PRODUCT_PATTERN.search(p):
        return "ot"
    return acq_brand(group)

def classify(order, history):
    date = _adt(_aval(order, "date", "timestamp"))
    amount = _amount(order)
    product = _product_text(order)
    status = _anorm(_aval(order, "status") or "")
    oid = str(order.get("id"))
    subs = [_sub(s) for s in history]
    this = next((s for s in subs if s["ref"] == oid), None)          # the subscription this order created...
    if this is None and date:                                        # ...or the one activated closest to it
        near = [s for s in subs if s["start"] and abs((s["start"] - date).days) <= 3]
        this = min(near, key=lambda s: abs(s["start"] - date), default=None)
    group = _aval(order, "group") or (this or {}).get("group")
    method = _aval(order, "paymentMethod") or (this or {}).get("method")
    brand = order_brand(group, product)
    prev = [s for s in subs if s is not this and s["start"] and s["start"] < date - dt.timedelta(days=1)
            and acq_brand(s["group"]) is not None and _code(s) != NOT_STARTED]
    paid_prev = [p for p in prev if not _is_free(p)]
    gap_days = None
    acq_source = _anorm(_aval(order, "acquisitionSource", "acquisition_source", "acquisitionsource") or "")
    if SIMPLE_RULE:
        if "fail" in status:
            kind = f"excluded: status {status}"
        elif USE_ACQUISITION_SOURCE and "renew" in acq_source:
            kind = "excluded: acquisition source renewal"
        elif amount == 0:
            kind = "excluded: paid 0"
        elif brand is None:
            kind = "excluded: donation / integrale"
        else:
            kind = "NEW: simple rule"
    else:
        if status and status != "paid":
            kind = f"excluded: status {status}"
        elif brand is None:
            kind = "excluded: donation / integrale"
        elif acq_platform(method) is None:
            kind = "excluded: free (staff) / test"
        elif amount == 0:
            kind = "excluded: paid 0 (free / staff)"
        elif OLD_SUB_EXTENSION_IS_RENEWAL and this and this["start"] and this["start"] < date - dt.timedelta(days=RENEWAL_GAP_DAYS):
            kind = "not new: renewal of an existing subscription"
        elif any(_is_free(p) and _valid_at(p, date) for p in prev):
            kind = "not new: still has a valid free account"
        elif not paid_prev:
            kind = "NEW: brand new" if not prev else "NEW: had only an expired free account"
        else:
            ends = [_ended_at(p) for p in paid_prev]
            running = [p for p, e in zip(paid_prev, ends) if e is None or e >= date]
            if BRANDS_SEPARATE:
                running = [p for p in running if acq_brand(p["group"]) == brand]
            past = [e for e in ends if e is not None and e < date]
            if running and not all(_cancelled_before(p, date) for p in running):
                kind = "not new: still subscribed (or changed plan)"
            elif running:
                kind = "NEW: had cancelled, came back before it ran out"
            elif not past:
                kind = "NEW: new to this brand (subscribes to the other one)"
            elif max(past) + dt.timedelta(days=LAPSE_DAYS) <= date:
                kind = f"NEW: came back after {LAPSE_DAYS}+ days"
            else:
                kind = f"not new: came back < {LAPSE_DAYS} days"
            if past and not running:
                gap_days = (date - max(past)).days
    return {"order_id": oid, "user_id": str(_aval(order, "user_id", "userId") or ""), "date": date,
            "week": _week_name(date), "group": group, "product": product, "brand": brand, "method": method,
            "platform": acq_platform(method) or "Web", "status": status, "acq_source": acq_source, "amount": amount, "matched_sub": (this or {}).get("id"), "kind": kind,
            "gap_days": gap_days}

def acquisitions(results, week):
    counts, seen = {"olj": Counter(), "ot": Counter()}, set()
    for x in sorted(results, key=lambda x: x["date"]):
        if x["week"] == week and x["kind"].startswith("NEW") and (SIMPLE_RULE or (x["brand"], x["user_id"]) not in seen):
            seen.add((x["brand"], x["user_id"]))
            counts[x["brand"]][x["platform"]] += 1
    out = {}
    for b in ("olj", "ot"):
        out[f"{b}_new_split"] = {p: counts[b].get(p, 0) for p in ("Web", "App")}
        out[f"{b}_new"] = sum(out[f"{b}_new_split"].values())
    return out

# ---------- The acquisitions sheet (Google Sheet, or the downloaded .xlsx as a fallback) ----------
def _parse_acq_sheet(raw):
    """raw: the tab as a DataFrame (no header). -> {'YYYY-MM-DD': {'olj': Basic + Premium, 'ot': OT}} for filled-in days."""
    head = raw.head(5).astype(str).apply(lambda col: col.str.strip().str.lower())
    def col_of(text):
        hits = [c for c in head.columns if head[c].str.contains(text, regex=False).any()]
        if not hits:
            raise ValueError(f"no '{text}' column")
        return hits[0]
    c_date, c_basic, c_prem, c_ot = col_of("date"), col_of("totales basic"), col_of("totales premium"), col_of("totales ot")
    detail = [c for c in raw.columns if c_date < c < min(c_basic, c_prem, c_ot)     # the per-offer columns (typed in),
              and not head[c].str.contains("total|acquisitions", regex=True).any()]   # not the formula totals
    num = lambda v: 0 if pd.isna(pd.to_numeric(v, errors="coerce")) else int(float(v))
    out = {}
    for _, r in raw.iterrows():
        v = r[c_date]
        d = (pd.Timestamp("1899-12-30") + pd.Timedelta(days=float(v))) if isinstance(v, (int, float)) and not pd.isna(v) \
            else pd.to_datetime(v, errors="coerce")
        if pd.isna(d) or all(pd.isna(pd.to_numeric(r[c], errors="coerce")) for c in detail):
            continue                                    # not a date, or a day nobody has filled in yet
        out[d.strftime("%Y-%m-%d")] = {"olj": num(r[c_basic]) + num(r[c_prem]), "ot": num(r[c_ot])}
    return out

def read_acq_sheet():
    """Daily totals from the acquisitions sheet; {} (and a printed reason) if it can't be read."""
    try:
        from google.oauth2 import service_account as _sa
        from googleapiclient.discovery import build as _build
        creds = _sa.Credentials.from_service_account_file(
            "service_account.json", scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"])
        api = _build("sheets", "v4", credentials=creds, cache_discovery=False).spreadsheets()
        tabs = [t["properties"] for t in api.get(spreadsheetId=ACQ_SHEET_ID, fields="sheets.properties").execute()["sheets"]]
        order = [t["title"] for t in tabs if t["sheetId"] == ACQ_SHEET_GID] + \
                [t["title"] for t in tabs if t["title"].strip().lower() == "acquisitions"]
        for title in dict.fromkeys(order):
            rows = api.values().get(spreadsheetId=ACQ_SHEET_ID, range=f"'{title}'", valueRenderOption="UNFORMATTED_VALUE",
                                    dateTimeRenderOption="SERIAL_NUMBER").execute().get("values", [])
            width = max((len(r) for r in rows), default=0)
            try:
                daily = _parse_acq_sheet(pd.DataFrame([r + [None] * (width - len(r)) for r in rows]))
                print(f"[sheet] read '{title}' from the Google Sheet: {len(daily)} filled-in days")
                return daily
            except ValueError as e:
                print(f"[sheet] tab '{title}' doesn't look like the acquisitions tab ({e})")
        print("[sheet] no acquisitions tab found in the Google Sheet")
    except Exception as e:
        print(f"[sheet] couldn't read the Google Sheet ({type(e).__name__}: {str(e)[:150]})")
    if Path(ACQ_SHEET_XLSX).exists():
        daily = _parse_acq_sheet(pd.read_excel(ACQ_SHEET_XLSX, sheet_name="Acquisitions", header=None))
        print(f"[sheet] using the uploaded {ACQ_SHEET_XLSX} instead: {len(daily)} filled-in days")
        return daily
    print("[sheet] -> the report will show the notebook's numbers without the sheet's ( ) totals")
    return {}

def _sheet_week_total(daily, start, end, b):
    days = [d.strftime("%Y-%m-%d") for d in pd.date_range(start, end)]
    return sum(daily[d][b] for d in days) if all(d in daily for d in days) else None   # None = week not complete in the sheet

# ---------- Run ----------
if not CMS_READY:
    raise RuntimeError("CMS not configured -- set CMS_API_KEY (⚙️ Settings / Colab secret / GitHub secret).")
if SIMPLE_RULE:
    print("[subscriptions] SIMPLE RULE: every 'New subscriptions' order counts, except status Fail, amount 0, "
          "donation / integrale in the product (brand: 'Today' / 'OT' in the product -> OT)")
else:
    print(f"[subscriptions] rule: NEW = brand new, or every previous subscription ended {LAPSE_DAYS}+ days before "
          f"(LAPSE_DAYS = {LAPSE_DAYS}) | old-subscription extensions = renewal: {OLD_SUB_EXTENSION_IS_RENEWAL}")
_orders = [o for o in fetch_new_orders() if _week_name(_adt(_aval(o, "date", "timestamp")))]
_hist = {}
for _uid in ([] if SIMPLE_RULE else sorted({str(_aval(o, "user_id", "userId") or "") for o in _orders} - {""})):
    _body = _acq_get(f"/cms/customer/{_uid}/subscription")
    _hist[_uid] = [s for s in (_body if isinstance(_body, list) else []) if isinstance(s, dict)]
ACQ_RESULTS = [classify(o, _hist.get(str(_aval(o, "user_id", "userId") or ""), [])) for o in _orders]

this_week, last_week = acquisitions(ACQ_RESULTS, "this"), acquisitions(ACQ_RESULTS, "last")
if SIMPLE_RULE and USE_ACQUISITION_SOURCE and not any(x["acq_source"] for x in ACQ_RESULTS):
    print("⚠️  [subscriptions] no 'acquisition source' came back from the API -> renewals could NOT be filtered out. "
          "Run the raw API test cell and tell me the field's real name.")
SHEET_DAILY = read_acq_sheet()
for _d, _s, _e in ((this_week, WEEK_START, WEEK_END), (last_week, PREV_WEEK_START, PREV_WEEK_END)):
    for _b in ("olj", "ot"):
        _d[f"{_b}_sheet"] = _sheet_week_total(SHEET_DAILY, _s, _e, _b)
print(f"[sheet] this week OLJ {this_week['olj_sheet']} / OT {this_week['ot_sheet']} | "
      f"last week OLJ {last_week['olj_sheet']} / OT {last_week['ot_sheet']}   (None = week not filled in)")
ACQUISITIONS_SOURCE = "CMS"

print(f"[subscriptions] {len(_orders)} new subscription orders, {len(_hist)} buyers checked")
for _k, _v in Counter(x["kind"] for x in ACQ_RESULTS).most_common():
    print(f"    {_v:4d}  {_k}")
for _w, _d in (("last", last_week), ("this", this_week)):
    try:
        _x = _acq_get("/cms/analytics/orders/count-by-payment-method",
                      [("date_from", f"{(PREV_WEEK_START if _w == 'last' else WEEK_START):%Y-%m-%d}"),
                       ("date_to", f"{(PREV_WEEK_END if _w == 'last' else WEEK_END):%Y-%m-%d}"), ("order_type", "2")])
        _cms = sum(int(v or 0) for t in _x.get("traces", []) for v in t.get("data", []))
    except Exception as e:
        _cms = f"n/a ({type(e).__name__})"
    _ours = sum(1 for x in ACQ_RESULTS if x["week"] == _w and x["status"] in ("", "paid"))   # paid orders only
    print(f"  {_w} week: OLJ {_d['olj_new']} (Web {_d['olj_new_split']['Web']} / App {_d['olj_new_split']['App']}) | "
          f"OT {_d['ot_new']} (Web {_d['ot_new_split']['Web']} / App {_d['ot_new_split']['App']}) | "
          f"check: orders {_ours} vs CMS {_cms}{'' if _ours == _cms else '  <- differ'}")

with open(ACQ_CSV, "w", newline="", encoding="utf-8") as _f:
    _wr = csv.DictWriter(_f, fieldnames=list(ACQ_RESULTS[0]) if ACQ_RESULTS else ["order_id"])
    _wr.writeheader(); _wr.writerows(ACQ_RESULTS)
print(f"  one line per order -> {ACQ_CSV}")

## 🔎 Check — why orders were labelled "still subscribed (or changed plan)" (optional, read-only)

Uses what Step 2 already fetched (no extra CMS calls). For each such order it shows the earlier subscription that blocked it, whether that subscription is the same brand, and why the notebook treated it as still running.


In [ ]:
# Read-only check on the orders labelled "not new: still subscribed (or changed plan)" -- no extra CMS calls.
# For each one it shows WHICH earlier subscription blocked it and WHY the notebook thinks it was still running.
_by_oid = {str(o.get("id")): o for o in _orders}
_chk = []
for x in ACQ_RESULTS:
    if not x["kind"].startswith("not new: still subscribed"):
        continue
    date = x["date"]
    subs = [_sub(s) for s in _hist.get(x["user_id"], [])]
    this = next((s for s in subs if s["id"] == str(x["matched_sub"])), None)
    prev = [s for s in subs if s is not this and s["start"] and s["start"] < date - dt.timedelta(days=1)
            and acq_brand(s["group"]) is not None and _code(s) != NOT_STARTED]
    for p in (p for p in prev if not _is_free(p)):
        e = _ended_at(p)
        if e is not None and e < date:
            continue                                            # this one had ended -> not the blocker
        if p["end"]:
            why = "ends AFTER the order (real overlap / plan change)"
        elif _code(p) in ACTIVE_CODES and p["renew"] and p["renew"] < date:
            why = f"NO end date, status {_code(p)}, next renewal already PASSED -> probably ended"
        elif _code(p) in ACTIVE_CODES:
            why = f"NO end date, status {_code(p)} (active), renews after the order"
        elif _code(p) in ENDED_CODES:
            why = f"NO end date, status {_code(p)} (ended), still valid until next renewal AFTER the order"
        else:
            why = f"NO end date, status {_code(p)} not in ACTIVE/ENDED lists -> treated as running"
        _chk.append({"week": x["week"], "order_id": x["order_id"], "user_id": x["user_id"], "order_date": date,
                     "order_brand": x["brand"], "order_group": x["group"],
                     "blocker_sub": p["id"], "blocker_brand": acq_brand(p["group"]), "blocker_group": p["group"],
                     "same_brand": acq_brand(p["group"]) == x["brand"], "blocker_status": _code(p),
                     "blocker_start": p["start"], "blocker_end": p["end"], "blocker_next_renewal": p["renew"],
                     "why": why})

_chk_df = pd.DataFrame(_chk)
_n = sum(1 for x in ACQ_RESULTS if x["kind"].startswith("not new: still subscribed"))
print(f"[check] {_n} 'still subscribed' orders; blockers found: {len(_chk_df)}")
if len(_chk_df):
    _one = _chk_df.sort_values("same_brand").drop_duplicates("order_id")   # one line per order
    print("\nWhy (one per order):")
    print(_one.groupby(["why", "same_brand"]).size().rename("orders").to_string())
    print("\nBy week and brand:")
    print(_one.groupby(["week", "order_brand", "same_brand"]).size().rename("orders").to_string())
    print("\nAll blockers:")
    print(_chk_df.drop(columns=["user_id"]).to_string(index=False))
    _chk_df.to_csv("still_subscribed_check.csv", index=False)
    print("\n-> still_subscribed_check.csv")


## 📊 Compare with the acquisitions sheet — highlights the days that don't match (optional, read-only)

Run after Step 2 (which reads the acquisitions Google Sheet with the service account, or the uploaded .xlsx). Compares the notebook's new subscriptions with the sheet for every day of the two weeks (OLJ = Basic + Premium, and OT), highlights the mismatching days (red = sheet has more, yellow = notebook has more) and lists the orders that could explain each one. Saves `acquisitions_vs_sheet.xlsx`. No extra CMS calls; skipped on GitHub.


In [ ]:
# Compares the notebook's NEW subscriptions with the acquisitions sheet, day by day, and highlights the days
# that don't match -- with the orders that could explain each one. Read-only: no extra CMS calls, nothing is sent.
# How to use: run it after Step 2, which reads the acquisitions Google Sheet (or the uploaded .xlsx).
# It covers the same two weeks as the report (last week + this week).
from IPython.display import display

if os.environ.get("GITHUB_ACTIONS") == "true":
    print("[compare] skipped on GitHub")
elif "ACQ_RESULTS" not in globals():
    print("[compare] run Step 2 first")
elif not globals().get("SHEET_DAILY"):
    print("[compare] the acquisitions sheet couldn't be read in Step 2 (see its [sheet] lines)")
else:
    _sheet = SHEET_DAILY
    _days = [d.strftime("%Y-%m-%d") for d in pd.date_range(PREV_WEEK_START, WEEK_END)]

    # notebook side: exactly like acquisitions() -- NEW only, one per person per brand per week, first order wins
    _counted, _seen = set(), set()
    for x in sorted(ACQ_RESULTS, key=lambda x: x["date"]):
        if x["week"] and x["kind"].startswith("NEW") and (SIMPLE_RULE or (x["brand"], x["user_id"], x["week"]) not in _seen):
            _seen.add((x["brand"], x["user_id"], x["week"])); _counted.add(x["order_id"])
    _res = pd.DataFrame(ACQ_RESULTS)
    _res["day"] = pd.to_datetime(_res["date"]).dt.strftime("%Y-%m-%d")
    _res["counted"] = _res["order_id"].isin(_counted)

    _rows, _cands = [], []
    for b, label in (("olj", "OLJ"), ("ot", "OT")):
        for d in _days:
            nb_n = int(((_res["brand"] == b) & (_res["day"] == d) & _res["counted"]).sum())
            sh_n = _sheet.get(d, {}).get(b)
            _rows.append({"Brand": label, "Day": d, "Weekday": pd.Timestamp(d).strftime("%a"),
                          "Sheet": sh_n, "Notebook": nb_n, "Notebook − Sheet": None if sh_n is None else nb_n - sh_n})
            if sh_n is None or sh_n == nb_n:
                continue
            day_orders = _res[(_res["brand"] == b) & (_res["day"] == d)]
            if sh_n > nb_n:     # sheet has more -> one of the orders the notebook LEFT OUT is probably in the sheet
                pick, why = day_orders[~day_orders["counted"]], f"sheet +{sh_n - nb_n}: not counted by the notebook"
            else:               # notebook has more -> one of the orders it COUNTED is probably missing from the sheet
                pick, why = day_orders[day_orders["counted"]], f"notebook +{nb_n - sh_n}: counted by the notebook"
            if pick.empty:
                _cands.append({"Brand": label, "Day": d, "Mismatch": why, "order_id": "—",
                               "kind": "no such order in the CMS that day -> check the sheet's entry"})
            for _, o in pick.sort_values("date").iterrows():
                _cands.append({"Brand": label, "Day": d, "Mismatch": why, "order_id": o["order_id"],
                               "user_id": o["user_id"], "time": pd.Timestamp(o["date"]).strftime("%H:%M"),
                               "method": o["method"], "amount": o.get("amount"), "group": o["group"],
                               "kind": o["kind"], "gap_days": o.get("gap_days")})

    _daily = pd.DataFrame(_rows)
    _bad = _daily[_daily["Notebook − Sheet"].fillna(0) != 0]
    _nodata = _daily["Sheet"].isna().sum()
    print(f"[compare] {len(_daily) - len(_bad) - _nodata} of {len(_daily)} brand-days match"
          f"{f' ({_nodata} days missing from the sheet)' if _nodata else ''}")
    for b in ("OLJ", "OT"):
        t = _daily[_daily["Brand"] == b]
        tw = t[t["Day"] >= f"{WEEK_START:%Y-%m-%d}"]; lw = t[t["Day"] < f"{WEEK_START:%Y-%m-%d}"]
        print(f"  {b}: last week sheet {int(lw['Sheet'].fillna(0).sum())} vs notebook {lw['Notebook'].sum()} | "
              f"this week sheet {int(tw['Sheet'].fillna(0).sum())} vs notebook {tw['Notebook'].sum()}")

    def _hl(row):
        diff = row["Notebook − Sheet"]
        color = "" if pd.isna(diff) or diff == 0 else ("background-color:#F4CCCC" if diff < 0 else "background-color:#FCE8B2")
        return [color] * len(row)
    _styled = _daily.style.apply(_hl, axis=1).format({"Sheet": lambda v: "—" if pd.isna(v) else f"{int(v)}",
                                                      "Notebook − Sheet": lambda v: "" if pd.isna(v) else f"{int(v):+d}"})
    print("\nRed = the sheet has more · yellow = the notebook has more")
    display(_styled)

    _cand_df = pd.DataFrame(_cands)
    if len(_cand_df):
        print("\nOrders to check on the mismatching days:")
        display(_cand_df)
    try:
        with pd.ExcelWriter("acquisitions_vs_sheet.xlsx") as _xw:
            _styled.to_excel(_xw, sheet_name="Daily comparison", index=False)
            _cand_df.to_excel(_xw, sheet_name="Orders to check", index=False)
        print("\n-> acquisitions_vs_sheet.xlsx (📁 Files panel, right-click -> Download)")
    except Exception as e:
        _daily.to_csv("acquisitions_vs_sheet_daily.csv", index=False); _cand_df.to_csv("acquisitions_vs_sheet_orders.csv", index=False)
        print(f"\n-> saved as CSV instead ({type(e).__name__}): acquisitions_vs_sheet_daily.csv, acquisitions_vs_sheet_orders.csv")


## 🧪 Side test — acquisitions day by day (optional, not part of the report)

Counts each day's acquisitions under several possible definitions (all orders, new orders, excluding donations/free/test, and the "genuinely new" rule with different lapse days) so they can be compared with the acquisitions sheet. Type the sheet's daily numbers into `SHEET` and it ranks which definition matches best. Off by default (`RUN_SIDE_TEST = False`) and it never runs on GitHub, so the weekly report is unaffected. Read-only: nothing is changed or sent.


In [ ]:
from collections import defaultdict

RUN_SIDE_TEST = False   # <-- set to True, then run this cell (after Step 2). Never runs on GitHub.

if not RUN_SIDE_TEST or os.environ.get("GITHUB_ACTIONS") == "true":
    print("[test] side test skipped (set RUN_SIDE_TEST = True to run it)")
else:

    # ---- 1. Which days to test ----------------------------------------------------------------
    TEST_START = dt.datetime(2026, 9, 7)
    TEST_END   = dt.datetime(2026, 9, 20)

    # ---- 2. (optional) The acquisitions sheet's numbers, per day ------------------------------
    # Type them in and the script says which definition matches the sheet best. Leave {} to skip.
    # Filled in from Daily_sheet_Acquisitions_Churns.xlsx, "Acquisitions" tab (OLJ = Basic + Premium), 1-24 Sep 2026.
    # Only the days between TEST_START and TEST_END are compared.
    SHEET = {
        "OLJ": {"2026-09-01": 11, "2026-09-02": 4, "2026-09-03": 6, "2026-09-04": 2, "2026-09-05": 3, "2026-09-06": 1, "2026-09-07": 12, "2026-09-08": 13, "2026-09-09": 11, "2026-09-10": 5, "2026-09-11": 6, "2026-09-12": 6, "2026-09-13": 7, "2026-09-14": 13, "2026-09-15": 16, "2026-09-16": 9, "2026-09-17": 8, "2026-09-18": 8, "2026-09-19": 3, "2026-09-20": 7, "2026-09-21": 8, "2026-09-22": 11, "2026-09-23": 7, "2026-09-24": 7},
        "OT":  {"2026-09-01": 4, "2026-09-02": 3, "2026-09-03": 1, "2026-09-04": 5, "2026-09-05": 1, "2026-09-06": 0, "2026-09-07": 4, "2026-09-08": 5, "2026-09-09": 5, "2026-09-10": 1, "2026-09-11": 6, "2026-09-12": 0, "2026-09-13": 3, "2026-09-14": 5, "2026-09-15": 2, "2026-09-16": 3, "2026-09-17": 5, "2026-09-18": 4, "2026-09-19": 6, "2026-09-20": 0, "2026-09-21": 7, "2026-09-22": 3, "2026-09-23": 6, "2026-09-24": 4},
    }

    LAPSE_OPTIONS = [0, 7, 30, 60, 90]   # "came back after N days" values to try

    # ---- Fetch every PAID SUBSCRIPTION order in the range (renewals included, tagged) ----------
    def _fetch_orders(start, end):
        base = [("status", "paid"), ("date_operator", "between"),
                ("date[start]", f"{start:%Y-%m-%d}"), ("date[end]", f"{end:%Y-%m-%d}"),
                ("hasRecurrentItems", "1")] + \
               [("columns[]", c) for c in ("id", "user_id", "group", "paymentMethod", "date", "status", "referenceOrderId", "amount", "products")]
        rows, seen, page = [], set(), None
        for _ in range(CMS_MAX_PAGES):
            body = _acq_get("/cms/payment", base + ([("page", page)] if page else []))
            fresh = [o for o in (body.get("data") or []) if str(o.get("id")) not in seen]
            if not fresh:
                break
            seen.update(str(o.get("id")) for o in fresh); rows += fresh
            page = (body["page"] if body.get("page") is not None else (page or 0)) + 1
        return rows

    print(f"[test] fetching paid subscription orders {TEST_START:%Y-%m-%d} -> {TEST_END:%Y-%m-%d} ...")
    _t_orders = []
    for o in _fetch_orders(TEST_START, TEST_END):
        d = _adt(_aval(o, "date", "timestamp"))
        if d and TEST_START.date() <= d.date() <= TEST_END.date():
            _t_orders.append(o)

    _t_hist = {}
    for uid in sorted({str(_aval(o, "user_id", "userId") or "") for o in _t_orders} - {""}):
        if uid in globals().get("_hist", {}):
            _t_hist[uid] = _hist[uid]                          # already fetched by Step 2
        else:
            b = _acq_get(f"/cms/customer/{uid}/subscription")
            _t_hist[uid] = [s for s in (b if isinstance(b, list) else []) if isinstance(s, dict)]
    print(f"[test] {len(_t_orders)} orders, {len(_t_hist)} buyers")

    # ---- Label every order under every definition --------------------------------------------
    _saved = (LAPSE_DAYS, OLD_SUB_EXTENSION_IS_RENEWAL)
    _detail = []
    try:
        for o in _t_orders:
            uid = str(_aval(o, "user_id", "userId") or "")
            hist = _t_hist.get(uid, [])
            base = classify(o, hist)
            row = {"day": base["date"].date().isoformat(), "order_id": base["order_id"], "user_id": uid,
                   "brand": base["brand"], "platform": base["platform"], "group": base["group"], "method": base["method"],
                   "is_renewal_order": bool(str(_aval(o, "referenceOrderId") or "").strip()),
                   "excluded": base["kind"].startswith("excluded")}
            for ext in (True, False):
                for lapse in LAPSE_OPTIONS:
                    LAPSE_DAYS, OLD_SUB_EXTENSION_IS_RENEWAL = lapse, ext
                    row[f"new_lapse{lapse}{'' if ext else '_noext'}"] = classify(o, hist)["kind"].startswith("NEW")
            LAPSE_DAYS, OLD_SUB_EXTENSION_IS_RENEWAL = _saved
            row["kind_now"] = base["kind"]
            _detail.append(row)
    finally:
        LAPSE_DAYS, OLD_SUB_EXTENSION_IS_RENEWAL = _saved

    # ---- The candidate definitions -----------------------------------------------------------
    VARIANTS = {
        "A all paid sub orders (incl. renewals)": lambda r: True,
        "B new orders (no referenceOrderId)":      lambda r: not r["is_renewal_order"],
        "C B minus donation/free/test":            lambda r: not r["is_renewal_order"] and not r["excluded"],
    }
    for ext in (True, False):
        for lapse in LAPSE_OPTIONS:
            key = f"new_lapse{lapse}{'' if ext else '_noext'}"
            label = f"D NEW, lapse {lapse}d" + ("" if ext else ", old-sub extensions count")
            VARIANTS[label] = (lambda k: lambda r: not r["is_renewal_order"] and r[k])(key)

    def _daily(brand, unique_users):
        days = pd.date_range(TEST_START, TEST_END).strftime("%Y-%m-%d")
        table = {}
        for name, ok in VARIANTS.items():
            per_day = defaultdict(set if unique_users else list)
            for r in _detail:
                if (brand is None or r["brand"] == brand) and ok(r):
                    (per_day[r["day"]].add if unique_users else per_day[r["day"]].append)(r["user_id"] if unique_users else r["order_id"])
            table[name] = [len(per_day.get(d, ())) for d in days]
        return pd.DataFrame(table, index=days).T

    # ---- Print -----------------------------------------------------------------------------
    pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
    for brand, label in (("olj", "OLJ"), ("ot", "OT")):
        for uniq in (False, True):
            t = _daily(brand, uniq)
            t["TOTAL"] = t.sum(axis=1)
            print(f"\n===== {label} -- {'unique buyers' if uniq else 'orders'} per day =====")
            print(t.to_string())

        sheet = {k: v for k, v in (SHEET.get(label) or {}).items()
                 if TEST_START.date() <= dt.date.fromisoformat(k) <= TEST_END.date()}
        if sheet:
            print(f"\n----- {label}: closest match to the sheet -----")
            scores = []
            for uniq in (False, True):
                t = _daily(brand, uniq)
                for name, vals in t.iterrows():
                    diffs = [abs(vals.get(d, 0) - n) for d, n in sheet.items()]
                    scores.append((sum(diffs), -sum(x == 0 for x in diffs),
                                   f"{name} [{'unique buyers' if uniq else 'orders'}]"))
            for total, neg_exact, name in sorted(scores)[:5]:
                print(f"  off by {total:3d} in total, exact on {-neg_exact}/{len(sheet)} days  ->  {name}")

    pd.DataFrame(_detail).to_csv("acquisitions_daily_detail.csv", index=False)
    print("\n[test] one line per order, with every definition's yes/no -> acquisitions_daily_detail.csv")


## 3. GA4 — shared query functions

Read-only GA4 Data API, logged in with the Google Cloud service account (`service_account.json`; on GitHub it's
written from the `GSHEET_SA_KEY` secret — the name is historical). Every query **must** be given a brand's
stream filter:

| Brand | GA4 streams |
|---|---|
| OLJ | stream name contains "olj" |
| OT | stream name starts with "OT" or contains "orient today" (the web stream is "L'Orient Today") |

If GA4 isn't connected, each function returns clearly fake **example data**, so the rest of the notebook
can still be checked.

One-time setup: enable the Google Analytics Data API in the service account's project, then add the service
account's email as **Viewer** under GA4 Admin → Property Access Management. As a fallback, an OAuth
`oauth_client_secret.json` (Desktop app) logs in as you once and reuses `token.json` after that.

In [ ]:
SERVICE_ACCOUNT_FILE = "service_account.json"  # Google Cloud service account key (GitHub: written from the GSHEET_SA_KEY secret)
GA4_PROPERTY_ID = "328439412"  # GA4 Admin > Property Details
OAUTH_CLIENT_SECRET_FILE = "oauth_client_secret.json"  # from Cloud Console > Credentials > OAuth client ID (Desktop app)
OAUTH_TOKEN_FILE = "token.json"  # created automatically after your first approval; reused silently after that
GA4_SCOPES = ["https://www.googleapis.com/auth/analytics.readonly"]

def get_ga4_credentials():
    """Prefers the service account (once Property Access Management is granted); falls back to
    logging in as you via OAuth, which only needs a browser click on the very first run."""
    if os.path.exists(SERVICE_ACCOUNT_FILE):
        from google.oauth2 import service_account as ga_service_account
        return ga_service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=GA4_SCOPES)

    if os.path.exists(OAUTH_CLIENT_SECRET_FILE):
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from google_auth_oauthlib.flow import InstalledAppFlow

        creds = None
        if os.path.exists(OAUTH_TOKEN_FILE):
            creds = Credentials.from_authorized_user_file(OAUTH_TOKEN_FILE, GA4_SCOPES)
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(OAUTH_CLIENT_SECRET_FILE, GA4_SCOPES)
                creds = flow.run_local_server(port=0)  # opens your browser -- click Allow once
            with open(OAUTH_TOKEN_FILE, "w") as f:
                f.write(creds.to_json())
        return creds

    return None

GA4_READY = (
    (os.path.exists(SERVICE_ACCOUNT_FILE) or os.path.exists(OAUTH_CLIENT_SECRET_FILE))
    and GA4_PROPERTY_ID != "REPLACE_WITH_YOUR_PROPERTY_ID"
)

# Always defined (even as None when GA4 isn't connected yet), so downstream cells can safely pass
# stream_filter=olj_stream_filter / ot_stream_filter regardless of GA4_READY -- avoids a NameError
# in the example-data fallback path, where these never get built.
olj_stream_filter = None
ot_stream_filter = None

if GA4_READY:
    from google.analytics.data_v1beta import BetaAnalyticsDataClient
    from google.analytics.data_v1beta.types import (
        RunReportRequest, DateRange, Metric, Dimension, OrderBy,
        FilterExpression, Filter,
    )
    ga4_creds = get_ga4_credentials()
    ga4_client = BetaAnalyticsDataClient(credentials=ga4_creds)

    # OLJ streams (name contains "olj",
        # case-insensitive, so it matches "OLJ Website" / "olj android" / "OLJ iOS" etc.)
    olj_stream_filter = FilterExpression(
        filter=Filter(
            field_name="streamName",
            string_filter=Filter.StringFilter(
                match_type=Filter.StringFilter.MatchType.CONTAINS,
                value="olj",
                case_sensitive=False,
            )
        )
    )


    # OT streams don't share one consistent naming pattern -- "OT iOS"/"OT Android" start with
    # "OT", but the web stream is apparently named "L'Orient Today" with no "OT" in it at all.
    # A bare CONTAINS "ot" would be too loose (matches almost anything with those two letters
    # together), so this ORs two specific, safe patterns instead: starts with "OT", OR contains
    # "orient today".
    from google.analytics.data_v1beta.types import FilterExpressionList
    ot_stream_filter = FilterExpression(
        or_group=FilterExpressionList(expressions=[
            FilterExpression(filter=Filter(
                field_name="streamName",
                string_filter=Filter.StringFilter(
                    match_type=Filter.StringFilter.MatchType.BEGINS_WITH,
                    value="OT",
                    case_sensitive=False,
                )
            )),
            FilterExpression(filter=Filter(
                field_name="streamName",
                string_filter=Filter.StringFilter(
                    match_type=Filter.StringFilter.MatchType.CONTAINS,
                    value="orient today",
                    case_sensitive=False,
                )
            )),
        ])
    )

    def ga4_report(start, end, metrics, dimensions=None, order_by_metric=None, limit=None,
                   extra_filter=None, stream_filter=None):
        if stream_filter is None:
            raise ValueError("ga4_report needs a brand's stream_filter (see the Brands section)")
        base_filter = stream_filter
        dim_filter = base_filter
        if extra_filter is not None:
            from google.analytics.data_v1beta.types import FilterExpressionList
            dim_filter = FilterExpression(
                and_group=FilterExpressionList(expressions=[base_filter, extra_filter])
            )
        request = RunReportRequest(
            property=f"properties/{GA4_PROPERTY_ID}",
            date_ranges=[DateRange(start_date=start.strftime("%Y-%m-%d"), end_date=end.strftime("%Y-%m-%d"))],
            metrics=[Metric(name=m) for m in metrics],
            dimensions=[Dimension(name=d) for d in (dimensions or [])],
            dimension_filter=dim_filter,
            limit=limit,
        )
        if order_by_metric:
            request.order_bys = [OrderBy(metric=OrderBy.MetricOrderBy(metric_name=order_by_metric), desc=True)]
        resp = ga4_client.run_report(request)
        cols = [d.name for d in resp.dimension_headers] + [m.name for m in resp.metric_headers]
        rows = []
        for row in resp.rows:
            rows.append([v.value for v in row.dimension_values] + [v.value for v in row.metric_values])
        return pd.DataFrame(rows, columns=cols)

    print(f"GA4 ready -- property {GA4_PROPERTY_ID}")
else:
    print("[GA4 not connected yet -- using example data for this section. "
          "Set GA4_PROPERTY_ID and make sure service_account.json is present + authorized on the property.]")

### 3a. Scorecard — Users, Sessions, Page views (Web / App / Total)

In [ ]:
# --- Web vs App split (used by every GA4 table below) ---
# GA4's `platform` dimension is "web", "iOS" or "Android" -> web = Web, iOS + Android = App.
PLATFORMS = ["Web", "App"]
SCORE_METRICS = {"users": "totalUsers", "sessions": "sessions", "page_views": "screenPageViews"}

def platform_bucket(p):
    return "Web" if str(p).strip().lower() == "web" else "App"

def ga4_scorecard(start, end, stream_filter=None):
    """{'Web': {...}, 'App': {...}, 'Total': {...}}, each with users / sessions / page_views."""
    if GA4_READY:
        mets = list(SCORE_METRICS.values())
        split = ga4_report(start, end, metrics=mets, dimensions=["platform"], stream_filter=stream_filter)
        split[mets] = split[mets].astype(int)
        by_bucket = split.assign(bucket=split["platform"].map(platform_bucket)).groupby("bucket")[mets].sum()
        total_row = ga4_report(start, end, metrics=mets, stream_filter=stream_filter).iloc[0]  # deduped total
        out = {b: {k: int(by_bucket.at[b, m]) if b in by_bucket.index else 0 for k, m in SCORE_METRICS.items()}
               for b in PLATFORMS}
        out["Total"] = {k: int(total_row[m]) for k, m in SCORE_METRICS.items()}
        return out
    else:
        # Example data
        base = 42000 if start == WEEK_START else 39900
        web_share = 0.64 if start == WEEK_START else 0.61
        total = {"users": base, "sessions": int(base * 1.39), "page_views": int(base * 2.66)}
        web = {k: int(v * web_share) for k, v in total.items()}
        app = {k: v - web[k] for k, v in total.items()}
        app["users"] = int(app["users"] * 1.06)  # some web/app overlap -> Web + App users > Total
        return {"Web": web, "App": app, "Total": total}

def pct_change(this_v, last_v):
    """Raw % change (None when last week was 0) -- the report colors arrows from this number."""
    if not last_v:
        return None
    return (this_v - last_v) / last_v * 100

GA4_ROWS = [("users", "Users"), ("sessions", "Sessions"), ("page_views", "Page views")]

### 3b. Top 10 articles (web + app views summed by Article ID, with a Web / App split)

Web events write the article ID to the `articleid` custom dimension; app events write it to a
**separate** `article_id` dimension — so this pulls both independently (each filtered to the brand's
streams) and merges them by matching ID value, summing views across web + app for the same article
before ranking. `platform` is pulled in the same query, so each article also gets its
Web and App views as separate columns (ranking is still by the total).

Also pulls `sessionSource` in the same query, so each article gets its **top 2 traffic sources**
shown as a percentage of that article's own views (e.g. "google 54% · (direct) 22%") — not a
share of site-wide traffic.

In [ ]:
import re

ARTICLE_ID_PATTERN = re.compile(r"^\d{6,7}$")  # valid article IDs only -- drops "(not set)" and junk

def _top2_sources(rows):
    """rows: list of (source, views) for one article. Returns two 'source NN%' strings, where the
    percentage is that source's share of THIS article's views (not overall site traffic)."""
    total = sum(v for _, v in rows)
    if total == 0:
        return "", ""
    ranked = sorted(rows, key=lambda t: t[1], reverse=True)[:2]
    out = [f"{src} {round(v / total * 100)}%" for src, v in ranked]
    while len(out) < 2:
        out.append("")
    return out[0], out[1]

def ga4_top_articles_by_id(start, end, n=10, stream_filter=None):
    if GA4_READY:
        # pageTitle + sessionSource pulled in the SAME query as the ID -- not a separate lookup call
        web = ga4_report(start, end, metrics=["screenPageViews"],
                          dimensions=["customEvent:articleid", "sessionSource", "pageTitle", "platform"], stream_filter=stream_filter)
        app = ga4_report(start, end, metrics=["screenPageViews"],
                          dimensions=["customEvent:article_id", "sessionSource", "pageTitle", "platform"], stream_filter=stream_filter)

        web = web.rename(columns={"customEvent:articleid": "article_id", "screenPageViews": "views"})
        app = app.rename(columns={"customEvent:article_id": "article_id", "screenPageViews": "views"})

        combined = pd.concat([web, app], ignore_index=True)
        combined["views"] = combined["views"].astype(int)

        # Keep only valid numeric article IDs (6 or 7 digits) -- excludes "(not set)", blanks, junk
        valid = combined["article_id"].astype(str).str.strip().apply(lambda v: bool(ARTICLE_ID_PATTERN.match(v)))
        combined = combined[valid]

        # Views per article split by platform (web -> Web, iOS/Android -> App), ranked by the total
        combined["bucket"] = combined["platform"].map(platform_bucket)
        totals = combined.pivot_table(index="article_id", columns="bucket", values="views",
                                      aggfunc="sum", fill_value=0).reindex(columns=PLATFORMS, fill_value=0)
        totals.columns.name = None
        totals["views"] = totals["Web"] + totals["App"]
        top = totals.sort_values("views", ascending=False).head(n).reset_index()

        # Rank by ID first (above), THEN attach a title per ID -- the most frequent title seen for
        # that ID across both web and app rows (guards against a stray differently-formatted title)
        subset = combined[combined["article_id"].isin(top["article_id"])]
        titles = (
            subset.groupby("article_id")["pageTitle"]
            .agg(lambda s: s.value_counts().idxmax() if len(s) else "")
            .to_dict()
        )

        # Top 2 traffic sources per article, as a share of that article's own views
        source_totals = subset.groupby(["article_id", "sessionSource"])["views"].sum()
        src1, src2 = {}, {}
        for aid in top["article_id"]:
            rows = list(source_totals.loc[aid].items()) if aid in source_totals.index.get_level_values(0) else []
            s1, s2 = _top2_sources(rows)
            src1[aid], src2[aid] = s1, s2

        top["Article"] = top["article_id"].map(titles)
        top["Top source 1"] = top["article_id"].map(src1)
        top["Top source 2"] = top["article_id"].map(src2)
        return top.rename(columns={"article_id": "Article ID", "views": "Views"})[
            ["Article", "Article ID", "Web", "App", "Views", "Top source 1", "Top source 2"]
        ]
    else:
        example_sources = [
            ("google 54%", "(direct) 22%"), ("facebook 41%", "google 30%"),
            ("(direct) 38%", "newsletter 19%"), ("google 47%", "instagram 15%"),
            ("(direct) 33%", "google 28%"), ("google 39%", "bing 12%"),
            ("newsletter 44%", "(direct) 21%"), ("google 36%", "facebook 20%"),
            ("(direct) 29%", "google 24%"), ("google 31%", "(direct) 26%"),
        ][:n]
        views = sorted([4200, 3100, 2650, 2200, 1800, 1600, 1400, 1250, 1100, 950], reverse=True)[:n]
        web = [int(v * s) for v, s in zip(views, [0.62, 0.55, 0.70, 0.48, 0.66, 0.59, 0.73, 0.51, 0.64, 0.57])]
        return pd.DataFrame({
            "Article": [f"Example article title {i}" for i in range(1, n + 1)],
            "Article ID": [f"15481{i:02d}" for i in range(1, n + 1)],
            "Web": web,
            "App": [v - w for v, w in zip(views, web)],
            "Views": views,
            "Top source 1": [s[0] for s in example_sources],
            "Top source 2": [s[1] for s in example_sources],
        })

### 3c. Top countries — share of each platform's sessions, WoW on session counts

In [ ]:
def ga4_split_by(start, end, dimension, metric="sessions", stream_filter=None):
    """<dimension> x platform, pivoted to one row per value with Web / App / Total counts.
    Sessions add up cleanly across platforms, so Total = Web + App here."""
    df = ga4_report(start, end, metrics=[metric], dimensions=[dimension, "platform"], limit=100000,
                    stream_filter=stream_filter)
    if df.empty:
        return pd.DataFrame(columns=PLATFORMS + ["Total"], dtype=int)
    df[metric] = df[metric].astype(int)
    df["bucket"] = df["platform"].map(platform_bucket)
    out = df.pivot_table(index=dimension, columns="bucket", values=metric, aggfunc="sum", fill_value=0)
    out = out.reindex(columns=PLATFORMS, fill_value=0)
    out.columns.name = None
    out["Total"] = out["Web"] + out["App"]
    return out.sort_values("Total", ascending=False)

def _share(num, den):
    return f"{num / den * 100:.0f}%" if den else "—"

def split_share_table(this_counts, last_counts, labels, label_col):
    """One row per label. For each of Total / Web / App: '<g> last' and '<g> this' = the label's share
    of that platform's sessions, '<g> WoW' = % change in the label's session COUNT (not the share)."""
    groups = ["Total"] + PLATFORMS
    this_tot = this_counts[groups].sum()
    last_tot = last_counts[groups].sum()
    rows = []
    for lab in labels:
        t = this_counts.loc[lab, groups] if lab in this_counts.index else pd.Series(0, index=groups)
        l = last_counts.loc[lab, groups] if lab in last_counts.index else None
        row = {label_col: lab}
        for g in groups:
            row[f"{g} last"] = _share(l[g], last_tot[g]) if l is not None else "—"
            row[f"{g} this"] = _share(t[g], this_tot[g])
            row[f"{g} WoW"] = pct_change(int(t[g]), int(l[g])) if l is not None else None
        rows.append(row)
    return pd.DataFrame(rows)

def country_counts(start, end, stream_filter):
    if GA4_READY:
        return ga4_split_by(start, end, "country", stream_filter=stream_filter)
    # Example data (Web, App sessions)
    if start == WEEK_START:
        data = {"Lebanon": (14800, 10000), "France": (6600, 2300), "USA": (4300, 1900),
                "Canada": (2000, 1100), "UAE": (1300, 1100), "Germany": (900, 300), "Belgium": (700, 200)}
    else:
        data = {"Lebanon": (13600, 9500), "France": (7000, 2400), "USA": (4000, 1800),
                "UAE": (1500, 1100), "Canada": (1500, 700), "Germany": (850, 300), "Belgium": (650, 200)}
    df = pd.DataFrame.from_dict(data, orient="index", columns=PLATFORMS)
    df["Total"] = df["Web"] + df["App"]
    return df.sort_values("Total", ascending=False)

def top_countries_with_wow(stream_filter, n=5):
    this_c = country_counts(WEEK_START, WEEK_END, stream_filter)
    last_c = country_counts(PREV_WEEK_START, PREV_WEEK_END, stream_filter)
    return split_share_table(this_c, last_c, list(this_c.head(n).index), "Country")

### 3d. Sources mapped into categories — Web / App / Total

Shared lists (direct, search engines, AI assistants) plus each brand's own social and newsletter lists.
They differ in two places: WhatsApp is `cms-46` for OLJ and `cms-48` for OT, and OT's Morning Brief /
"À la une" widgets are `cms-34` / `cms-9`.

In [ ]:
MAIN_CATS = ["Direct", "Search Engines", "Social Networks", "AI Assistants", "Internal/Newsletters", "Other"]

MAIN_COLORS = {
    "Direct":               "4285F4",
    "Search Engines":       "00BFA5",
    "Social Networks":      "9C27B0",
    "AI Assistants":        "FFA726",
    "Internal/Newsletters": "FF6B9D",
    "Other":                "9E9E9E",
}

DIRECT = {"(direct)"}

SEARCH_ENGINES = {
    "google", "news.google.com", "bing", "ecosia.org", "qwant.com", "duckduckgo",
    "fr.search.yahoo.com", "yahoo", "search.brave.com", "yandex", "ya.ru", "startpage.com",
}

AI_ASSISTANTS = {
    "chatgpt.com", "perplexity.ai", "gemini.google.com", "perplexity", "copilot.com",
    "copilot.microsoft.com", "openai", "duck.ai", "chat.mistral.ai",
    "notebooklm.google.com", "claude.ai", "poe.com", "grok.com", "chat.qwen.ai",
    "doubao.com", "chat.z.ai", "felo.ai", "mammouth.ai", "you.com",
}

# --- Brand-specific sets (passed in through each Brand) ---
# OLJ
SOCIAL_NETWORKS_OLJ = {
    "m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com",
    "mobile.facebook.com", "facebook", "l.instagram.com", "ig", "instagram",
    "instagram.com", "later-linkinbio", "linkin.bio", "t.co", "linkedin.com",
    "lnkd.in", "go.bsky.app", "l.threads.com", "twitter", "x.com", "threads",
    "bluesky", "pinterest.com", "tiktok.com", "snapchat", "snapchat.com",
    "web.whatsapp.com", "cms-46", "reddit.com", "fb", "flipboard", "flipboard.com",
}

INTERNAL_NEWSLETTERS_OLJ = {
    "mailchimp", "newsletter", "email", "website", "olj", "google-play", "olj.me",
    "olj.wael", "autopromoolj", "morningbrief", "hs_email", "brevo", "mailchi.mp",
    "us1.campaign-archive.com", "actito.be", "activetrail", "acumbamail", "omnisend",
    "wordfly", "newsletter_1", "newsletter_6b", "newsletter_paiementechouepp",
    "newsletter_preventif", "partenairesjamhour", "marketo",
    "gmi mailchimp integration prod list", "master list", "bundle", "nb",
}

# OT
SOCIAL_NETWORKS_OT = {
    "m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com",
    "mobile.facebook.com", "facebook", "l.instagram.com", "ig", "instagram",
    "instagram.com", "later-linkinbio", "linkin.bio", "t.co", "linkedin.com",
    "lnkd.in", "go.bsky.app", "l.threads.com", "twitter", "x.com", "threads",
    "bluesky", "pinterest.com", "tiktok.com", "snapchat", "snapchat.com",
    "web.whatsapp.com", "cms-48", "reddit.com", "old.reddit.com", "out.reddit.com",
    "fb", "flipboard", "flipboard.com",
    # Note: WhatsApp is "cms-48" for OT (it was "cms-46" for OLJ).
}

INTERNAL_NEWSLETTERS_OT = {
    "mailchimp", "newsletter", "email", "website", "ot", "olj", "google-play",
    "ot.me", "olj.me", "autopromoot", "autopromoolj", "morningbrief", "hs_email",
    "brevo", "mailchi.mp", "us1.campaign-archive.com", "actito.be", "activetrail",
    "acumbamail", "omnisend", "wordfly", "newsletter_1", "newsletter_6b",
    "newsletter_paiementechouepp", "newsletter_preventif", "partenairesjamhour",
    "marketo", "gmi mailchimp integration prod list", "master list", "bundle", "nb",
    # CMS-34 = Morning Brief, CMS-9 = A la une (both OT-owned newsletter/homepage widgets):
    "cms-34", "cms-9",
}

def categorize_source(src, social, newsletters):
    s = str(src).strip().lower()
    if s in DIRECT:
        return "Direct"
    if s in SEARCH_ENGINES:
        return "Search Engines"
    if s in social:
        return "Social Networks"
    if s in AI_ASSISTANTS:
        return "AI Assistants"
    if s in newsletters:
        return "Internal/Newsletters"
    return "Other"

def ga4_sources_by_category(start, end, stream_filter, social, newsletters):
    """Sessions per source category, split Web / App / Total (index = MAIN_CATS)."""
    if GA4_READY:
        # Every distinct source (no top-N slice) so categorization is complete
        by_src = ga4_split_by(start, end, "sessionSource", stream_filter=stream_filter)
        cats = by_src.index.map(lambda s: categorize_source(s, social, newsletters))
        return by_src.groupby(cats).sum().reindex(MAIN_CATS, fill_value=0)
    # Example data (Web, App sessions) -- varies by week so WoW has something to show
    if start == WEEK_START:
        data = {"Direct": (8400, 6800), "Search Engines": (11900, 1800), "Social Networks": (6900, 2500),
                "AI Assistants": (1100, 100), "Internal/Newsletters": (5600, 3800), "Other": (2300, 800)}
    else:
        data = {"Direct": (7900, 6200), "Search Engines": (10700, 1600), "Social Networks": (7600, 2600),
                "AI Assistants": (820, 80), "Internal/Newsletters": (5200, 3500), "Other": (2100, 800)}
    df = pd.DataFrame.from_dict(data, orient="index", columns=PLATFORMS).reindex(MAIN_CATS, fill_value=0)
    df["Total"] = df["Web"] + df["App"]
    return df

def sources_by_category_with_wow(stream_filter, social, newsletters):
    this_df = ga4_sources_by_category(WEEK_START, WEEK_END, stream_filter, social, newsletters)
    last_df = ga4_sources_by_category(PREV_WEEK_START, PREV_WEEK_END, stream_filter, social, newsletters)
    out = split_share_table(this_df, last_df, MAIN_CATS, "Category")
    out["Color"] = out["Category"].map(MAIN_COLORS)
    return out

### 3e. App downloads — iOS vs Android

In [ ]:
def ga4_app_downloads(start, end, stream_filter=None):
    if GA4_READY:
        from google.analytics.data_v1beta.types import Filter as _Filter, FilterExpression as _FE
        event_filter = _FE(filter=_Filter(field_name="eventName",
                                           string_filter=_Filter.StringFilter(value="app_download")))
        df = ga4_report(start, end, metrics=["eventCount"], dimensions=["platform"],
                         extra_filter=event_filter, stream_filter=stream_filter)
        counts = {row["platform"]: int(row["eventCount"]) for _, row in df.iterrows()}
        return {"ios": counts.get("iOS", 0), "android": counts.get("Android", 0)}
    else:
        return {"ios": 520, "android": 370} if start == WEEK_START else {"ios": 480, "android": 337}

## 4. Brands — the only brand-specific code

Each `Brand` holds what differs between the two briefs. Everything below takes a brand as input, so adding
a brand means adding one entry here (and its GA4 filter and source lists above).

`collect_brand_data` then pulls that brand's GA4 numbers and picks its CMS figures from steps 1–2.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Brand:
    """Everything that differs between the OLJ and OT briefs."""
    key: str                      # prefix used by the CMS results: "olj" / "ot"
    name: str                     # shown in the brief, the file name and the email subject
    stream_filter: object         # GA4 filter selecting this brand's streams
    social_sources: frozenset     # sessionSource values counted as Social Networks
    newsletter_sources: frozenset # sessionSource values counted as Internal/Newsletters
    recipients_variable: str      # GitHub variable that overrides SCHEDULED_RECIPIENTS[name]

    def file_stem(self):
        return f"Weekly_Analytics_Brief_{self.name}_{WEEK_START:%Y%m%d}_{WEEK_END:%Y%m%d}"

ALL_BRANDS = {
    "OLJ": Brand("olj", "OLJ", olj_stream_filter, frozenset(SOCIAL_NETWORKS_OLJ), frozenset(INTERNAL_NEWSLETTERS_OLJ),
                 "BRIEF_RECIPIENTS_OLJ"),
    "OT":  Brand("ot", "OT", ot_stream_filter, frozenset(SOCIAL_NETWORKS_OT), frozenset(INTERNAL_NEWSLETTERS_OT),
                 "BRIEF_RECIPIENTS_OT"),
}
BRANDS = [ALL_BRANDS[n] for n in BUILD_BRANDS]

@dataclass
class BrandData:
    """One brand's numbers for the week -- the only input the report, export and email steps need."""
    ga4_this: dict
    ga4_last: dict
    articles: pd.DataFrame
    countries: pd.DataFrame
    sources: pd.DataFrame
    downloads_this: dict
    downloads_last: dict
    accounts_this: dict
    accounts_last: dict
    subs_this: dict
    subs_last: dict

def collect_brand_data(brand):
    f = brand.stream_filter
    return BrandData(
        ga4_this=ga4_scorecard(WEEK_START, WEEK_END, stream_filter=f),
        ga4_last=ga4_scorecard(PREV_WEEK_START, PREV_WEEK_END, stream_filter=f),
        articles=ga4_top_articles_by_id(WEEK_START, WEEK_END, stream_filter=f),
        countries=top_countries_with_wow(f),
        sources=sources_by_category_with_wow(f, brand.social_sources, brand.newsletter_sources),
        downloads_this=ga4_app_downloads(WEEK_START, WEEK_END, stream_filter=f),
        downloads_last=ga4_app_downloads(PREV_WEEK_START, PREV_WEEK_END, stream_filter=f),
        accounts_this=this_week_accounts, accounts_last=last_week_accounts,   # both brands, from step 1
        subs_this=this_week, subs_last=last_week,                              # both brands, from step 2
    )

BRAND_DATA = {}
for b in BRANDS:
    BRAND_DATA[b.name] = collect_brand_data(b)
    d = BRAND_DATA[b.name]
    print(f"[{b.name}] users {d.ga4_this['Total']['users']:,} | new accounts {d.accounts_this[f'{b.key}_new_accounts']} "
          f"| new subscriptions {d.subs_this[f'{b.key}_new']}")

## 5. Final report — one brief per brand (preview)

`build_report(brand, data)` turns a brand's numbers into the brief once. The preview here and the Word export
in step 6 are both drawn from that same result, with the same colors, so what you see is what gets sent.

Layout (one A4 page): every comparison table has three color blocks, **WEB** (blue) · **APP** (green) ·
**TOTAL** (charcoal, last), each with Last wk · This wk · WoW. "One thing to watch" is auto-flagged from any
metric down 10%+ week over week.

In [ ]:
from IPython.display import HTML, display
import html as _html

# ---------- Look & feel (shared by this preview AND the Word export) ----------
GROUP_ORDER = ["Web", "App", "Total"]  # Total last, so it reads as Web + App = Total
GROUP_STYLE = {                          # head = block header, sub = sub-header, cell = body tint, text = accent
    "Total": {"head": "2B3440", "sub": "DDE1E6", "cell": "F3F4F6", "text": "2B3440"},
    "Web":   {"head": "1A73E8", "sub": "D6E4FB", "cell": "F0F5FE", "text": "1A5BB8"},
    "App":   {"head": "1E8E3E", "sub": "D3ECD9", "cell": "F0F8F2", "text": "17702F"},
}
INK, SOFT, MUTED = "2B3440", "5F6368", "8A9096"
UP_COLOR, DOWN_COLOR = "137333", "C5221F"
WATCH_ACCENT, WATCH_FILL = "E37400", "FEF4E6"
SUB_HEADERS = ["Last wk", "This wk", "WoW"]
TITLE_CLIP = 80

WEEK_LINE = (f"Week of {WEEK_START:%b %d} – {WEEK_END:%b %d, %Y}   ·   "
             f"compared with {PREV_WEEK_START:%b %d} – {PREV_WEEK_END:%b %d}")
FOOTNOTE = ("* New accounts: Total is higher than Web + App because it also includes Other sign-ups (CMS-9 and other "
            "non-web / non-app sources, e.g. newsletters) · "
            "** New subscriptions: the figure in ( ) comes from the acquisitions sheet. The old and new CMS don't fully "
            "agree, and Client Services reconciles them manually, so the two numbers can differ · "
            "Web = website · App = iOS + Android · Total users are de-duplicated, so Web + App users can exceed Total · "
            "Country / source figures are each platform's share of its own sessions; WoW is the change in sessions · "
            + (f"New subscriptions: CMS 'New subscriptions' orders{' with acquisition source New' if USE_ACQUISITION_SOURCE else ''}, "
               "excluding failed, 0-amount, donation and Intégrale orders" if SIMPLE_RULE else
               f"New subscriptions: paid orders from brand-new subscribers or people whose last subscription ended {LAPSE_DAYS}+ "
               "days earlier; staff, donations and renewals excluded")
            + "; App = Apple / Google Play.")

def fmt_wow(p):
    if p is None or pd.isna(p):
        return "n/a"
    return f"{'▲' if p >= 0 else '▼'} {abs(p):.0f}%"

def wow_color(p):
    if p is None or pd.isna(p):
        return MUTED
    return UP_COLOR if p >= 0 else DOWN_COLOR

def clip(s, n=TITLE_CLIP):
    s = str(s)
    return s if len(s) <= n else s[: n - 1].rstrip() + "…"

def cmp_counts(this_v, last_v):
    return (f"{last_v:,}", f"{this_v:,}", pct_change(this_v, last_v))

def share_rows(df, label_col):
    return [{"label": r[label_col], **{g: (r[f"{g} last"], r[f"{g} this"], r[f"{g} WoW"]) for g in GROUP_ORDER}}
            for _, r in df.iterrows()]

def new_accounts_row(brand, t, l):
    """Web / App split from the CMS source; 'Other' (newsletters etc.) is in Total and noted under the label."""
    k = brand.key
    row = {"label": "New accounts*", "Web": None, "App": None,   # * -> explained at the start of FOOTNOTE
           "Total": cmp_counts(t[f"{k}_new_accounts"], l[f"{k}_new_accounts"])}
    ts, ls = t.get(f"{k}_split"), l.get(f"{k}_split")
    if ts and ls:
        row["Web"] = cmp_counts(ts["Web"], ls["Web"])
        row["App"] = cmp_counts(ts["App"], ls["App"])
        if ts["Other"] or ls["Other"]:
            row["note"] = f"incl. {ts['Other']:,} other · {ls['Other']:,} last wk"
    return row

def new_subs_row(brand, t, l):
    """Total shows the acquisitions sheet's figure in ( ) when the sheet has that whole week, e.g. 62 (64)."""
    k = brand.key
    last_s, this_s, wow = cmp_counts(t[f"{k}_new"], l[f"{k}_new"])
    if l.get(f"{k}_sheet") is not None:
        last_s += f" ({l[f'{k}_sheet']:,})"
    if t.get(f"{k}_sheet") is not None:
        this_s += f" ({t[f'{k}_sheet']:,})"
    has_sheet = t.get(f"{k}_sheet") is not None or l.get(f"{k}_sheet") is not None
    row = {"label": f"New {brand.name} subscriptions" + ("**" if has_sheet else ""),   # ** -> explained in FOOTNOTE
           "Web": None, "App": None, "Total": (last_s, this_s, wow)}
    ts, ls = t.get(f"{k}_new_split"), l.get(f"{k}_new_split")
    if ts and ls:
        row["Web"] = cmp_counts(ts["Web"], ls["Web"])
        row["App"] = cmp_counts(ts["App"], ls["App"])
    return row

def watch_line_for(brand, d):
    """Metrics down 10%+ WoW (totals first; a platform only when the total hides it)."""
    tw, lw = d.ga4_this, d.ga4_last
    dl_this, dl_last = sum(d.downloads_this.values()), sum(d.downloads_last.values())
    movers = {label: pct_change(tw["Total"][k], lw["Total"][k]) for k, label in GA4_ROWS}
    movers["App downloads"] = pct_change(dl_this, dl_last)
    lines = [f"{name} down {abs(p):.0f}% week-over-week." for name, p in movers.items() if p is not None and p <= -10]
    for k, label in GA4_ROWS:
        if movers[label] is None or movers[label] > -10:
            for p in PLATFORMS:
                pc = pct_change(tw[p][k], lw[p][k])
                if pc is not None and pc <= -10:
                    lines.append(f"{p} {label.lower()} down {abs(pc):.0f}% week-over-week.")
    subs_now, subs_before = d.subs_this[f"{brand.key}_new"], d.subs_last[f"{brand.key}_new"]
    if subs_now < subs_before:
        lines.append(f"New {brand.name} subscriptions down {subs_before - subs_now} vs. last week ({subs_now} this week).")
    return " ".join(lines) if lines else "Nothing off-trend this week."

def _sources_cell(row):
    return row["Top source 1"] + (f" · {row['Top source 2']}" if row["Top source 2"] else "")

def build_report(brand, d):
    """Everything the preview, the Word export and the email need for one brand."""
    dl_this, dl_last = sum(d.downloads_this.values()), sum(d.downloads_last.values())
    scorecard = [
        {"label": label, **{g: cmp_counts(d.ga4_this[g][k], d.ga4_last[g][k]) for g in GROUP_ORDER}}
        for k, label in GA4_ROWS
    ] + [
        new_accounts_row(brand, d.accounts_this, d.accounts_last),
        new_subs_row(brand, d.subs_this, d.subs_last),
        {"label": "App downloads", "note": f"iOS {d.downloads_this['ios']} · Android {d.downloads_this['android']}",
         "Web": None, "Total": cmp_counts(dl_this, dl_last), "App": cmp_counts(dl_this, dl_last)},
    ]
    articles = [
        {"rank": i + 1, "title": row["Article"], "Total": f"{row['Views']:,}", "Web": f"{row['Web']:,}",
         "App": f"{row['App']:,}", "sources": _sources_cell(row)}
        for i, row in d.articles.reset_index(drop=True).iterrows()
    ]
    return {
        "brand": brand,
        "scorecard": ("Metric", scorecard),
        "countries": ("Country", share_rows(d.countries, "Country")),
        "sources": ("Source category", share_rows(d.sources, "Category")),
        "articles": articles,
        "watch": watch_line_for(brand, d),
    }

# ---------- Notebook preview (HTML, same colors as the .docx) ----------
def _h(s):
    return _html.escape(str(s))

_SEP = "border-left:3px solid #fff;"
def _td(content, style=""):
    return f'<td style="padding:3px 7px;border-bottom:1px solid #E6E8EB;white-space:nowrap;{style}">{content}</td>'

def _html_group_cells(cell, g):
    st = GROUP_STYLE[g]
    base = f"background:#{st['cell']};text-align:center;"
    if cell is None:
        return "".join(_td("—", base + f"color:#{MUTED};" + (_SEP if i == 0 else "")) for i in range(3))
    last, this, p = cell
    tot = g == "Total"
    return (_td(_h(last), base + _SEP + f"color:#{SOFT};font-weight:{600 if tot else 400};")
            + _td(_h(this), base + f"color:#{INK};font-weight:{800 if tot else 600};")
            + _td(fmt_wow(p), base + f"color:#{wow_color(p)};font-weight:700;"))

def html_cmp_table(label_header, rows):
    top = (f'<th rowspan="2" style="text-align:left;padding:4px 8px;color:#{INK};'
           f'border-bottom:2px solid #{INK}">{_h(label_header)}</th>')
    sub = ""
    for g in GROUP_ORDER:
        st = GROUP_STYLE[g]
        top += (f'<th colspan="3" style="background:#{st["head"]};color:#fff;padding:4px;font-size:11px;'
                f'letter-spacing:.08em;{_SEP}">{g.upper()}</th>')
        sub += "".join(f'<th style="background:#{st["sub"]};color:#{st["text"]};padding:3px 7px;font-size:10.5px;'
                       f'{_SEP if i == 0 else ""}">{s}</th>' for i, s in enumerate(SUB_HEADERS))
    body = ""
    for r in rows:
        label = _h(r["label"]) + (f'<div style="color:#{MUTED};font-size:10px;font-weight:400">{_h(r["note"])}</div>'
                                   if r.get("note") else "")
        body += "<tr>" + _td(label, f"text-align:left;color:#{INK};font-weight:600;") + "".join(
            _html_group_cells(r.get(g), g) for g in GROUP_ORDER) + "</tr>"
    return (f'<table style="border-collapse:collapse;width:100%;font-size:12px">'
            f'<tr>{top}</tr><tr>{sub}</tr>{body}</table>')

def html_articles_table(rows):
    neutral = f'text-align:left;padding:4px 8px;color:#{INK};border-bottom:2px solid #{INK};'
    head = (f'<th style="{neutral}">#</th><th style="{neutral}">Article</th>'
            + "".join(f'<th style="background:#{GROUP_STYLE[g]["head"]};color:#fff;padding:4px 8px;font-size:11px;'
                      f'letter-spacing:.08em;{_SEP}">{g.upper()}</th>' for g in GROUP_ORDER)
            + f'<th style="{neutral}{_SEP}">Top sources</th>')
    body = ""
    for r in rows:
        cells = _td(r["rank"], f"color:#{MUTED};") + _td(
            f'<span title="{_h(r["title"])}">{_h(clip(r["title"]))}</span>', f"color:#{INK};white-space:normal;")
        for g in GROUP_ORDER:
            cells += _td(r[g], f"background:#{GROUP_STYLE[g]['cell']};text-align:center;{_SEP}"
                              f"color:#{INK};font-weight:{800 if g == 'Total' else 500};")
        cells += _td(_h(r["sources"]), f"color:#{SOFT};font-size:11px;{_SEP}")
        body += f"<tr>{cells}</tr>"
    return f'<table style="border-collapse:collapse;width:100%;font-size:12px"><tr>{head}</tr>{body}</table>'

def _html_section(title, inner):
    return (f'<div style="font-size:11px;font-weight:700;letter-spacing:.1em;color:#{INK};margin:16px 0 6px">'
            f'{_h(title.upper())}</div>{inner}')

def _html_callout(label, text, accent, fill):
    return (f'<div style="background:#{fill};border-left:4px solid #{accent};padding:8px 12px;margin-top:12px;'
            f'border-radius:4px"><div style="font-size:10px;font-weight:700;letter-spacing:.1em;color:#{accent}">'
            f'{_h(label.upper())}</div><div style="font-size:13px;font-weight:600;color:#{INK};margin-top:2px">'
            f'{_h(text)}</div></div>')

def html_report(report):
    return (
        f'<div style="font-family:Calibri,Carlito,\'Segoe UI\',Arial,sans-serif;color:#{INK};background:#fff;'
        f'max-width:940px;padding:22px 26px;border:1px solid #E6E8EB;border-radius:10px;margin-bottom:18px">'
        f'<div style="font-size:24px;font-weight:800">Weekly Analytics Brief <span style="color:#{MUTED};'
        f'font-weight:600">· {_h(report["brand"].name)}</span></div>'
        f'<div style="font-size:12px;color:#{SOFT};margin-top:2px">{_h(WEEK_LINE)}</div>'
        + _html_section("Scorecard", html_cmp_table(*report["scorecard"]))
        + _html_section("Top 10 articles this week", html_articles_table(report["articles"]))
        + _html_section("Top countries", html_cmp_table(*report["countries"]))
        + _html_section("Sources by category", html_cmp_table(*report["sources"]))
        + _html_callout("One thing to watch", report["watch"], WATCH_ACCENT, WATCH_FILL)
        + f'<div style="font-size:10px;color:#{MUTED};margin-top:10px">{_h(FOOTNOTE)}</div></div>'
    )

if "BRAND_DATA" not in globals():
    raise RuntimeError("Run the earlier sections first (or Runtime > Run all).")
REPORTS = {name: build_report(ALL_BRANDS[name], d) for name, d in BRAND_DATA.items()}
for report in REPORTS.values():
    display(HTML(html_report(report)))

## 6. Export — one Word (+ PDF) per brand

PDF needs LibreOffice (installed on GitHub; skipped if missing). In Colab both files download for each brand.

In [ ]:
import shutil
import subprocess

from docx import Document
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm, Pt, RGBColor

PAGE_W, PAGE_H = 21.0, 29.7          # A4 portrait, cm
MARGIN_X, MARGIN_TOP, MARGIN_BOTTOM = 1.3, 1.1, 1.0
CONTENT_W = PAGE_W - 2 * MARGIN_X
BASE_PT = 8                          # body size; everything is sized so the brief fits on one page
DOWNLOAD_IN_COLAB = True

_ALIGN = {"left": WD_ALIGN_PARAGRAPH.LEFT, "center": WD_ALIGN_PARAGRAPH.CENTER}
_TCPR_ORDER = ["cnfStyle", "tcW", "gridSpan", "hMerge", "vMerge", "tcBorders", "shd", "noWrap", "tcMar",
               "textDirection", "tcFitText", "vAlign", "hideMark"]
_TBLPR_ORDER = ["tblStyle", "tblpPr", "tblOverlap", "bidiVisual", "tblStyleRowBandSize", "tblStyleColBandSize",
                "tblW", "jc", "tblCellSpacing", "tblInd", "tblBorders", "shd", "tblLayout", "tblCellMar", "tblLook"]

def _el(tag, **attrs):
    e = OxmlElement(tag)
    for k, v in attrs.items():
        e.set(qn(f"w:{k}"), str(v))
    return e

def _reorder(parent, order):
    """Word is strict about child order inside tcPr / tblPr -- sort what we appended into schema order."""
    kids = list(parent)
    rank = lambda e: order.index(e.tag.split("}")[1]) if e.tag.split("}")[1] in order else len(order)
    for k in kids:
        parent.remove(k)
    for k in sorted(kids, key=rank):
        parent.append(k)

def _shade(cell, fill):
    cell._tc.get_or_add_tcPr().append(_el("w:shd", val="clear", color="auto", fill=fill))

def _borders(cell, **edges):  # edge=(size in 1/8 pt, hex color)
    tcPr = cell._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = _el("w:tcBorders"); tcPr.append(b)
    for edge, (sz, color) in edges.items():
        b.append(_el(f"w:{edge}", val="single", sz=sz, space=0, color=color))

def _tight(p, before=0, after=0):
    pf = p.paragraph_format
    pf.space_before, pf.space_after, pf.line_spacing = Pt(before), Pt(after), 1.0

def _run(p, text, size=BASE_PT, bold=False, color=INK, italic=False):
    r = p.add_run(str(text))
    r.font.size, r.bold, r.italic = Pt(size), bold, italic
    r.font.color.rgb = RGBColor.from_string(color)
    return r

def _write(cell, text, *, size=BASE_PT, bold=False, color=INK, align="center", fill=None, note=None):
    cell.text = ""
    p = cell.paragraphs[0]
    _tight(p)
    p.alignment = _ALIGN[align]
    _run(p, text, size, bold, color)
    if note:
        _run(p, "", size).add_break()
        _run(p, note, size - 1.5, color=MUTED)
    if fill:
        _shade(cell, fill)
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

def _table(doc, nrows, widths_cm, rules=True):
    t = doc.add_table(rows=nrows, cols=len(widths_cm))
    t.autofit = False
    tblPr = t._tbl.tblPr
    tblW = tblPr.find(qn("w:tblW"))
    if tblW is None:
        tblW = _el("w:tblW"); tblPr.append(tblW)
    tblW.set(qn("w:type"), "dxa"); tblW.set(qn("w:w"), str(int(sum(widths_cm) * 567)))
    b = _el("w:tblBorders")  # light horizontal rules only -- no grid
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        on = rules and edge in ("bottom", "insideH")
        b.append(_el(f"w:{edge}", val="single" if on else "nil", sz=4, space=0, color="E3E6EA"))
    tblPr.append(b)
    mar = _el("w:tblCellMar")
    for edge, w in (("top", 22), ("left", 70), ("bottom", 22), ("right", 70)):
        mar.append(_el(f"w:{edge}", w=w, type="dxa"))
    tblPr.append(mar)
    for i, w in enumerate(widths_cm):
        t.columns[i].width = Cm(w)
        for c in t.columns[i].cells:
            c.width = Cm(w)
    return t

WHITE_SEP = (18, "FFFFFF")  # white gap between the Total / Web / App blocks

def docx_cmp_table(doc, label_header, rows, label_w=3.9):
    val_w = (CONTENT_W - label_w) / 9
    t = _table(doc, 2 + len(rows), [label_w] + [val_w] * 9)
    lab = t.cell(0, 0).merge(t.cell(1, 0))
    _write(lab, label_header, bold=True, align="left")
    _borders(lab, bottom=(10, INK))
    for gi, g in enumerate(GROUP_ORDER):
        st, c0 = GROUP_STYLE[g], 1 + 3 * gi
        h = t.cell(0, c0).merge(t.cell(0, c0 + 2))
        _write(h, g.upper(), size=BASE_PT - 0.5, bold=True, color="FFFFFF", fill=st["head"])
        _borders(h, left=WHITE_SEP)
        for i, s in enumerate(SUB_HEADERS):
            c = t.cell(1, c0 + i)
            _write(c, s, size=BASE_PT - 1, bold=True, color=st["text"], fill=st["sub"])
            if i == 0:
                _borders(c, left=WHITE_SEP)
    for ri, r in enumerate(rows, start=2):
        _write(t.cell(ri, 0), r["label"], bold=True, align="left", note=r.get("note"))
        for gi, g in enumerate(GROUP_ORDER):
            st, c0, cell, tot = GROUP_STYLE[g], 1 + 3 * gi, r.get(g), g == "Total"
            if cell is None:
                vals = [("—", MUTED, False)] * 3
            else:
                last, this, p = cell
                vals = [(last, SOFT, tot), (this, INK, True), (fmt_wow(p), wow_color(p), True)]
            for i, (text, color, bold) in enumerate(vals):
                c = t.cell(ri, c0 + i)
                _write(c, text, bold=bold, color=color, fill=st["cell"],
                       size=BASE_PT + (0.5 if tot and i == 1 else 0))
                if i == 0:
                    _borders(c, left=WHITE_SEP)
    return t

def docx_articles_table(doc, rows):
    num_w = 1.45
    widths = [0.55, 7.35, num_w, num_w, num_w]
    widths.append(CONTENT_W - sum(widths))
    t = _table(doc, 1 + len(rows), widths)
    for i, h in enumerate(["#", "Article"]):
        _write(t.cell(0, i), h, bold=True, align="left")
        _borders(t.cell(0, i), bottom=(10, INK))
    for gi, g in enumerate(GROUP_ORDER):
        c = t.cell(0, 2 + gi)
        _write(c, g.upper(), size=BASE_PT - 0.5, bold=True, color="FFFFFF", fill=GROUP_STYLE[g]["head"])
        _borders(c, left=WHITE_SEP)
    _write(t.cell(0, 5), "Top sources", bold=True, align="left")
    _borders(t.cell(0, 5), bottom=(10, INK), left=WHITE_SEP)
    for ri, r in enumerate(rows, start=1):
        _write(t.cell(ri, 0), r["rank"], color=MUTED, align="left")
        _write(t.cell(ri, 1), clip(r["title"]), align="left")
        for gi, g in enumerate(GROUP_ORDER):
            c = t.cell(ri, 2 + gi)
            _write(c, r[g], bold=(g == "Total"), fill=GROUP_STYLE[g]["cell"])
            _borders(c, left=WHITE_SEP)
        _write(t.cell(ri, 5), r["sources"], size=BASE_PT - 1, color=SOFT, align="left")
        _borders(t.cell(ri, 5), left=WHITE_SEP)
    return t

def docx_callout(doc, label, text, accent, fill):
    t = _table(doc, 1, [CONTENT_W], rules=False)
    c = t.cell(0, 0)
    _shade(c, fill)
    _borders(c, left=(28, accent))
    p = c.paragraphs[0]
    _tight(p)
    _run(p, label.upper(), BASE_PT - 1, True, accent)
    p2 = c.add_paragraph()
    _tight(p2)
    _run(p2, text, BASE_PT + 1.5, True, INK)

def docx_section(doc, title):
    p = doc.add_paragraph()
    _tight(p, before=9, after=3)
    _run(p, title.upper(), BASE_PT + 0.5, True, INK)

def build_brief_docx(report, path):
    doc = Document()
    sec = doc.sections[0]
    sec.page_width, sec.page_height = Cm(PAGE_W), Cm(PAGE_H)
    sec.left_margin = sec.right_margin = Cm(MARGIN_X)
    sec.top_margin, sec.bottom_margin = Cm(MARGIN_TOP), Cm(MARGIN_BOTTOM)
    normal = doc.styles["Normal"]
    normal.font.name, normal.font.size = "Calibri", Pt(BASE_PT)
    normal.element.get_or_add_rPr().get_or_add_rFonts().set(qn("w:eastAsia"), "Calibri")

    p = doc.paragraphs[0] if doc.paragraphs else doc.add_paragraph()
    _tight(p)
    _run(p, "Weekly Analytics Brief", 17, True, INK)
    _run(p, f"  ·  {report['brand'].name}", 17, True, MUTED)
    p = doc.add_paragraph()
    _tight(p, after=6)
    _run(p, WEEK_LINE, BASE_PT + 1, color=SOFT)

    docx_section(doc, "Scorecard")
    docx_cmp_table(doc, *report["scorecard"])
    docx_section(doc, "Top 10 articles this week")
    docx_articles_table(doc, report["articles"])
    docx_section(doc, "Top countries")
    docx_cmp_table(doc, *report["countries"])
    docx_section(doc, "Sources by category")
    docx_cmp_table(doc, *report["sources"])
    spacer = doc.add_paragraph()
    _tight(spacer, after=4)
    docx_callout(doc, "One thing to watch", report["watch"], WATCH_ACCENT, WATCH_FILL)
    p = doc.add_paragraph()
    _tight(p, before=5)
    _run(p, FOOTNOTE, BASE_PT - 1.5, color=MUTED)

    zoom = doc.settings.element.find(qn("w:zoom"))  # python-docx's template omits a required attribute
    if zoom is not None and zoom.get(qn("w:percent")) is None:
        zoom.set(qn("w:percent"), "100")

    body = doc.element.body
    for tcPr in body.iter(qn("w:tcPr")):
        _reorder(tcPr, _TCPR_ORDER)
    for tcb in body.iter(qn("w:tcBorders")):
        _reorder(tcb, ["top", "left", "bottom", "right", "insideH", "insideV"])
    for tblPr in body.iter(qn("w:tblPr")):
        _reorder(tblPr, _TBLPR_ORDER)
    doc.save(path)
    return path

def export_pdf(docx_path):
    """Optional PDF copy (handy for checking the one-page layout). Needs LibreOffice; skipped if absent."""
    soffice = shutil.which("soffice") or shutil.which("libreoffice")
    if not soffice:
        return None
    out_dir = os.path.dirname(os.path.abspath(docx_path))
    subprocess.run([soffice, "--headless", "--convert-to", "pdf", "--outdir", out_dir, docx_path],
                   check=True, capture_output=True, timeout=180)
    pdf_path = os.path.splitext(docx_path)[0] + ".pdf"
    return pdf_path if os.path.exists(pdf_path) else None

def export_brief(report):
    """Word + optional PDF for one brand; returns the file paths."""
    docx_path = build_brief_docx(report, report["brand"].file_stem() + ".docx")
    print(f"[{report['brand'].name}] saved {os.path.abspath(docx_path)}")
    pdf_path = None
    try:
        pdf_path = export_pdf(docx_path)
        print(f"[{report['brand'].name}] saved {os.path.abspath(pdf_path)}" if pdf_path else "  [PDF skipped -- LibreOffice not installed]")
    except Exception as e:
        print(f"  [PDF export failed, .docx is fine] {type(e).__name__}: {e}")
    return [p for p in (docx_path, pdf_path) if p]

EXPORTED_FILES = {name: export_brief(report) for name, report in REPORTS.items()}

if DOWNLOAD_IN_COLAB:
    try:
        from google.colab import files as colab_files
        for paths in EXPORTED_FILES.values():
            for f in paths:
                colab_files.download(f)
    except ImportError:
        pass  # not in Colab -- files are saved next to the notebook

## 7. Email — one per brand (only after you've checked the exports)

Asks for a **y/N confirmation** before sending when run by hand (sends right away once you say yes).

**Who gets it:** test runs (Colab, or a manual *Run workflow*) go only to `TEST_RECIPIENTS`; the
scheduled Monday send goes to each brand's `SCHEDULED_RECIPIENTS` list, or to the GitHub variable
`BRIEF_RECIPIENTS_OLJ` / `BRIEF_RECIPIENTS_OT` if set (both in ⚙️ Settings). Each brand gets its own email with
only its own brief attached.

**Scheduled run:** `.github/workflows/weekly_brief.yml` runs this notebook every **Monday
morning** (Beirut time, summer and winter handled) with `SEND_EMAIL=1`. When the notebook finishes, this
cell **waits until 12:00** and sends both emails, so the email always lands at 12:00 even if the run took a few
minutes. Change the time with `SEND_AT` below; the workflow starts about 35 minutes earlier.

**Gmail setup (one-time — App Password, not your regular password):**
1. The sending account needs **2-Step Verification** turned on (myaccount.google.com/security).
2. Go to **myaccount.google.com/apppasswords** → create one for "Mail" → copy the 16-character password.
3. Set `GMAIL_ADDRESS` and `GMAIL_APP_PASSWORD` as environment variables, or it'll prompt you.

⚠️ Untested against a real send from here. If Gmail rejects the login, double-check it's an
**App Password** (normal passwords are rejected once 2-Step Verification is on).

In [ ]:
import getpass
import time
import smtplib
from email import encoders
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

SEND_AT = dt.time(12, 0)          # Beirut time. Scheduled (GitHub) runs wait until then before sending
SEND_TZ = ZoneInfo("Asia/Beirut")
IS_SCHEDULED = os.environ.get("GITHUB_EVENT_NAME") == "schedule"   # set by GitHub Actions itself
IN_CI = os.environ.get("GITHUB_ACTIONS") == "true"

def _split_emails(text):
    return [e.strip() for e in str(text).replace(";", ",").replace("\n", ",").split(",") if e.strip()]

def recipients_for(brand):
    """Test runs -> TEST_RECIPIENTS. Scheduled runs -> the brand's GitHub variable, else its settings list
    (for OLJ, the older BRIEF_RECIPIENTS variable still works too)."""
    if not IS_SCHEDULED:
        return list(TEST_RECIPIENTS)
    env = os.environ.get(brand.recipients_variable, "") or (os.environ.get("BRIEF_RECIPIENTS", "") if brand.name == "OLJ" else "")
    out = _split_emails(env) or list(SCHEDULED_RECIPIENTS.get(brand.name, []))
    bad = [e for e in out if "@" not in e]
    assert out and not bad, f"[{brand.name}] check the recipient list -- empty or invalid: {bad or out}"
    return out

def send_email_with_attachments(recipients, subject, body, attachment_paths, bcc=False):
    sender = os.environ.get("GMAIL_ADDRESS") or input("Sending Gmail address: ")
    app_password = os.environ.get("GMAIL_APP_PASSWORD") or getpass.getpass("Gmail App Password (not your normal password): ")

    msg = MIMEMultipart()
    msg["From"], msg["Subject"] = sender, subject
    msg["To"] = sender if bcc else ", ".join(recipients)   # BCC: addresses only go in the envelope below
    msg.attach(MIMEText(body, "plain"))
    for path in attachment_paths:
        with open(path, "rb") as f:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(f.read())
        encoders.encode_base64(part)
        part.add_header("Content-Disposition", f'attachment; filename="{os.path.basename(path)}"')
        msg.attach(part)

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(sender, app_password)
        server.send_message(msg, from_addr=sender, to_addrs=list(recipients))

def build_email(report, files):
    """Subject + body of one brand's weekly mail -- edit the wording here."""
    name = report["brand"].name
    fmt = "Word + PDF" if any(f.endswith(".pdf") for f in files) else "Word"
    subject = f"📊 {name} Weekly Brief · {WEEK_START:%b %d}–{WEEK_END:%b %d}"
    body = f"""Hello from the Direction Numérique 👋

This week's {name} Analytics Brief ({WEEK_START:%b %d} – {WEEK_END:%b %d, %Y}) is attached ({fmt}): one page, Web · App · Total.

🔎 One thing to watch: {report["watch"]}

🤖 This email is automated and sent every Monday at {SEND_AT:%H:%M}. Something looks off? Just let us know.

Direction Numérique · L'Orient-Le Jour
"""
    return subject, body

def wait_until_send_time():
    """Scheduled runs start a bit early (GitHub's scheduler can lag); hold the emails until SEND_AT."""
    now = dt.datetime.now(SEND_TZ)
    target = now.replace(hour=SEND_AT.hour, minute=SEND_AT.minute, second=0, microsecond=0)
    wait = (target - now).total_seconds()
    if 0 < wait <= 3 * 3600:
        print(f"Waiting until {SEND_AT:%H:%M} Beirut time to send ({wait / 60:.0f} min)...")
        time.sleep(wait)

def send_brief(report, files):
    brand = report["brand"]
    to = recipients_for(brand)
    subject, body = build_email(report, files)
    send_email_with_attachments(to, subject, body, files, bcc=EMAIL_AS_BCC)
    print(f"[{brand.name}] emailed to {len(to)} recipient(s): {', '.join(to)}" + ("  (as BCC)" if EMAIL_AS_BCC else ""))

# ---------- Which briefs to send ----------
SEND_EMAIL = os.environ.get("SEND_EMAIL", "").strip().lower() in ("1", "true", "yes")
if SEND_EMAIL:
    to_send = list(REPORTS)
elif IN_CI:
    to_send = []
else:                                   # by hand: preview each email, then ask per brand
    to_send = []
    for name, report in REPORTS.items():
        subj, body = build_email(report, EXPORTED_FILES[name])
        print(f"\n--- {name} email preview ---\nTo: {', '.join(recipients_for(report['brand']))}\nSubject: {subj}\n\n{body}")
        if input(f"Checked the {name} brief? Send it now? [y/N] ").strip().lower() == "y":
            to_send.append(name)

if to_send and IN_CI and os.environ.get("SEND_NOW") != "1":   # manual "send now" runs skip the wait
    wait_until_send_time()
for name in to_send:
    try:
        send_brief(REPORTS[name], EXPORTED_FILES[name])
    except Exception as e:
        print(f"[{name}] email step failed -- paste this error back and I'll adjust.\n{type(e).__name__}: {e}")
if not to_send:
    print("Not sent.")